# Anotación de Árboles Frutales

**SAM2 Fine-tuning + Clasificador de Copas + Auto-etiquetado por Finca**

---

1. **Setup**: Verificación de GPU e instalación de dependencias
2. **Configuración**: Definición de las 15 fincas y parámetros (incluye tamaño de SAM2, workers de descarga, límite de tiles del clasificador)
3. **Descarga CVAT**: Frames y anotaciones de las 15 fincas, en paralelo y con reintentos automáticos
4. **Análisis de etiquetado**: detección, finca por finca, de frames y shapes etiquetados vs. sin etiquetar (snapshot "antes")
5. **Preparación Dataset**: Conversión a formato SAM2 SA-V, usando los frames ya etiquetado de las 15 fincas
6. **Clasificador**: RandomForest (crop_tree vs background), con negativos auto-generados y métricas hold-out (precisión/recall/F1)
7. **Fine-tune SAM2**: Entrenamiento del modelo de segmentación, con validación de IoU por epoch
8. **Inferencia**: Auto-etiquetado de los frames sin etiquetar, finca por finca (segmentación binaria + componentes conexas)
9. **Upload**: Subida a CVAT vía el endpoint correcto (PATCH .../annotations?action=create), con prueba en 1 finca antes de subir todas
10. **Reporte final**: Consolidación de todas las métricas (antes/después)
```

---

## 1. Setup e Instalación de Dependencias

Verificar la GPU disponible e instalar todas las dependencias necesarias.
El notebook necesita: `torch`, `sam2`, `opencv`, `scikit-learn`, `pycocotools`.

In [ ]:
# Verificar GPU disponible
import torch

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("\n" + "="*60 + "\nGPU not available.\n" + \
                     "Runtime → Change runtime type → GPU T4.\n" + \
                     "="*60)

gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} (compute capability {cc[0]}.{cc[1]})")

if cc[0] < 7:
    raise SystemExit(f"GPU {gpu_name} incompatible. Requires T4 (sm_75).")

DEVICE = 'cuda'
print(f"Device: {DEVICE}")

In [ ]:
# Instalar dependencias necesarias
!pip install -q sam2 opencv-python-headless shapely scipy pycocotools tqdm requests
print("Dependencies installed successfully.")

---

## 2. Configuración del Proyecto

Definir las rutas, configuración de las fincas, task IDs de CVAT y credenciales de acceso.

In [ ]:
import os
from pathlib import Path

# Raíz del proyecto (compatible con Kaggle)
PROJECT_ROOT = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
print(f"Project root: {PROJECT_ROOT}")

# Opciones de ejecución
UPLOAD_TO_CVAT = False        # Cambiar a True solo tras revisar resultados
EPOCHS_SAM2 = 10              # Epochs para fine-tuning de SAM2
EPOCHS_CLF = 300              # Número de árboles en RandomForest
SPLIT_RATIO = 0.8             # Proporción train/val (80/20)
MIN_AREA_POLYGON = 900        # Área mínima de polígono en px para fincas sin GSD conocido
MIN_CROWN_DIAMETER_M = 1.2    # Diámetro mínimo de una copa de árbol (metros). Se convierte a píxeles según el GSD de cada finca
MAX_CROWN_DIAMETER_M = 8.0    # área máxima por de una copa de árbol (metros).
MAX_TILES_CLASSIFIER = None   # Límite de frames para extraer crops del clasificador.
DOWNLOAD_MAX_WORKERS = 4      # Descargas de frames en paralelo por finca
SAM2_MODEL_SIZE = "large"     # "large" (más preciso, default) | "base_plus" | "small" | "tiny"
CLEANUP_RAW_FRAMES_AFTER_USE = True  # Borra los frames PNG crudos de cada finca 
MAX_OVERLAYS_PER_FARM = 3            # Overlays de inferencia a guardar por finca

print(f"Upload to CVAT: {UPLOAD_TO_CVAT}")
print(f"SAM2 epochs: {EPOCHS_SAM2}")
print(f"Classifier epochs: {EPOCHS_CLF}")
print(f"Max classifier tiles: {MAX_TILES_CLASSIFIER if MAX_TILES_CLASSIFIER else 'no limit'}")
print(f"Parallel downloads: {DOWNLOAD_MAX_WORKERS}")
print(f"SAM2 model size: {SAM2_MODEL_SIZE}")
print(f"Cleanup raw frames after use: {CLEANUP_RAW_FRAMES_AFTER_USE}")
print(f"Max overlays per farm: {MAX_OVERLAYS_PER_FARM}")
print(f"Crown diameter range (GSD-aware area filter): {MIN_CROWN_DIAMETER_M}m - {MAX_CROWN_DIAMETER_M}m")

In [ ]:
# Configuración de las 15 fincas (tasks de CVAT)

FARMS = {
    "Casa_Luna":      {"task_id": 2476948, "label_id": 8155070, "crop_type": "Naranja",  "gsd_cm_px": 8.58},
    "Ciruelo":        {"task_id": 2476943, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 3.03},
    "Colmangos":      {"task_id": 2476935, "label_id": 8155070, "crop_type": "Mango",    "gsd_cm_px": 5.05},
    "Esparragos":     {"task_id": 2476930, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 3.12},
    "La_Carolina":    {"task_id": 2476922, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 6.58},
    "La_Manuela":     {"task_id": 2595405, "label_id": 8155070, "crop_type": "Naranja",  "gsd_cm_px": 7.50},
    "La_Montera":     {"task_id": 2475996, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 6.18},
    "La_Teresita":    {"task_id": 2595419, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 8.56},
    "Las_Villas":     {"task_id": 2475966, "label_id": 8155070, "crop_type": "Limon",    "gsd_cm_px": 3.58},
    "Magdala":        {"task_id": 2475942, "label_id": 8155070, "crop_type": "Limon",    "gsd_cm_px": 5.25},
    "Naranjo":        {"task_id": 2475936, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 2.81},
    "Playa_Rica":     {"task_id": 2475934, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 9.72},
    "Toronja":        {"task_id": 2475931, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 2.83},
    "Villa_Antonela": {"task_id": 2475927, "label_id": 8155070, "crop_type": "Aguacate", "gsd_cm_px": 2.72},
    "Yulima":         {"task_id": 2475910, "label_id": 8155070, "crop_type": "Limon",    "gsd_cm_px": 3.84},
}

print(f"Total configured farms: {len(FARMS)}")
for name, cfg in FARMS.items():
    print(f"  {name}: task {cfg['task_id']}, crop={cfg['crop_type']}, GSD={cfg['gsd_cm_px']}cm/px")

In [ ]:
# Credenciales CVAT
import os

os.environ["CVAT_HOST"] = "https://app.cvat.ai"

# Las credenciales solo se leen de Kaggle Secrets (Add-ons)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["CVAT_USER"] = secrets.get_secret("cvat_user")
    os.environ["CVAT_PASSWORD"] = secrets.get_secret("cvat_pass")
    print(f"CVAT credentials loaded from Kaggle Secrets: {os.environ['CVAT_USER'][:3]}***")
except Exception as e:
    raise RuntimeError(
        "Could not load CVAT credentials from Kaggle Secrets."
        f"Original error: {e}"
    )

print(f"Host: {os.environ['CVAT_HOST']}")

---

## 3. Descarga de Datos desde CVAT

Descarga anotaciones de las **15 fincas**, pero **solo los frames ya etiquetados**
(no los que aún no tienen anotación). Los frames sin etiquetar son la mayoría del
volumen y no se necesitan todavía — se descargan bajo demanda, finca por finca y
solo los que tocan, en la Sección 8 (Inferencia), y se borran apenas se usan.

Esto mantiene bajo el número de archivos crudos en disco durante toda la corrida:
lo único que se acumula de entrada es lo ya etiquetado (poco), no las 15 fincas
completas.

La función `download_cvat_data()` maneja:
- Autenticación con la API de CVAT
- Obtención de la lista completa de frames por finca (metadata liviana)
- Descarga de anotaciones (polígonos) ya existentes en CVAT
- Descarga como PNG **solo** de los frames con al menos un polígono anotado
- Generación de manifest.json por finca (incluye la lista completa de frame ids,
  para poder calcular cobertura sin necesitar los PNG sin etiquetar)


In [ ]:
import requests
import json
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

class CvatClient:
    """Cliente ligero para la API de CVAT (solo las funciones necesarias)."""

    def __init__(self, host, user, password, max_retries=5, backoff_base=1.5):
        self.host = host.rstrip('/')
        self.session = requests.Session()
        self.max_retries = max_retries
        self.backoff_base = backoff_base
        self._authenticate(user, password)

    def _request(self, method, url, **kwargs):
        last_error = None
        for attempt in range(1, self.max_retries + 1):
            try:
                resp = self.session.request(method, url, timeout=30, **kwargs)

                if resp.status_code == 429 and attempt < self.max_retries:
                    retry_after = resp.headers.get('Retry-After')
                    wait_time = float(retry_after) if retry_after else 5.0 * (2 ** (attempt - 1))
                    time.sleep(wait_time)
                    last_error = requests.exceptions.HTTPError(
                        f"429 Too Many Requests (esperó {wait_time:.1f}s antes de reintentar)"
                    )
                    continue

                resp.raise_for_status()
                return resp
            except requests.exceptions.RequestException as e:
                last_error = e
                if attempt < self.max_retries:
                    wait_time = self.backoff_base ** attempt
                    time.sleep(wait_time)
        raise last_error

    def _authenticate(self, user, password):
        """Autenticar con la API de CVAT y obtener cookies."""
        url = f"{self.host}/api/auth/login"
        resp = self._request('POST', url, json={"username": user, "password": password})
        print(f"Authentication successful: {user[:3]}***")

    def get_task_info(self, task_id):
        """Obtener metadata de una task."""
        url = f"{self.host}/api/tasks/{task_id}"
        resp = self._request('GET', url)
        return resp.json()

    def get_frames(self, task_id):
        url = f"{self.host}/api/tasks/{task_id}/data/meta"
        resp = self._request('GET', url)
        data = resp.json()
        frames_meta = data.get('frames', [])
        return [{'id': i, **fm} for i, fm in enumerate(frames_meta)]

    def download_frame(self, task_id, frame_id, output_path, quality='original'):
        """
        Descargar un frame como imagen.
        """
        url = f"{self.host}/api/tasks/{task_id}/data"
        params = {'type': 'frame', 'number': frame_id, 'quality': quality}
        resp = self._request('GET', url, params=params, stream=True)
        with open(output_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

    def download_frames_parallel(self, task_id, frames, frames_dir, max_workers=8, desc="Downloading frames"):
        """
        Descarga varios frames en paralelo (varios requests HTTP a la vez).
        """
        frames_dir = Path(frames_dir)
        frames_dir.mkdir(parents=True, exist_ok=True)

        def _download_one(frame):
            frame_id = frame['id']
            frame_path = frames_dir / f"frame_{frame_id:05d}.png"
            if frame_path.exists():
                return True
            try:
                self.download_frame(task_id, frame_id, frame_path)
                return True
            except Exception as e:
                print(f"    Error downloading frame {frame_id} (after retries): {e}")
                return False

        downloaded = 0
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(_download_one, f): f for f in frames}
            for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
                if fut.result():
                    downloaded += 1
        return downloaded

    def get_annotations(self, task_id):
        """Obtener anotaciones (polígonos) de una task."""
        url = f"{self.host}/api/tasks/{task_id}/annotations"
        resp = self._request('GET', url)
        return resp.json()

print("CvatClient class defined successfully.")

In [ ]:
def download_cvat_data(task_ids, output_dir, client, max_workers=8, persist_to_output=True):
    """
    Descarga anotaciones y frames etiquetados desde CVAT para las tasks especificadas.

    Por cada task:
    1. Obtiene metadata de la task
    2. Obtiene la lista completa de frames (metadata liviana, no descarga nada)
    3. Descarga anotaciones (polígonos)
    4. Descarga los frames que tienen al menos un polígono anotado
    5. Genera manifest.json (incluye la lista completa de frame ids, para que
       la Sección 4 pueda calcular cobertura sin necesitar los PNG sin etiquetar)
    6. Si persist_to_output=True, copia manifest.json + annotations.json
       (livianos) a /kaggle/output de inmediato.

    Args:
        task_ids: Lista de task IDs a descargar
        output_dir: Directorio raíz de salida (data/raw/)
        client: Instancia de CvatClient autenticado
        max_workers: Descargas de frames en paralelo por task
        persist_to_output: Copiar manifest/anotaciones a /kaggle/output tras cada task

    Returns:
        Diccionario con resumen por task
    """
    output_root = Path(output_dir)
    summary = {}

    kaggle_output_meta = Path('/kaggle/output/raw_meta')
    if persist_to_output:
        kaggle_output_meta.mkdir(parents=True, exist_ok=True)

    for task_id in task_ids:
        print(f"\n{'='*60}")
        print(f"Downloading labeled frames for task {task_id}...")
        print(f"{'='*60}")

        # Crear directorio para esta task
        task_dir = output_root / str(task_id)
        frames_dir = task_dir / "frames"
        frames_dir.mkdir(parents=True, exist_ok=True)

        # 1. Obtener metadata de la task
        try:
            info = client.get_task_info(task_id)
            name = info.get('name', f'task_{task_id}')
            print(f"  Name: {name}")
        except Exception as e:
            print(f"  Error getting info (after retries): {e}")
            continue

        # 2. Obtener lista completa de frames (solo metadata, aún no descarga nada)
        try:
            all_frames = client.get_frames(task_id)
            print(f"  Total frames in task: {len(all_frames)}")
        except Exception as e:
            print(f"  Error getting frames (after retries): {e}")
            continue

        # 3. Obtener anotaciones y determinar qué frame ids están etiquetados
        try:
            annotations = client.get_annotations(task_id)
            with open(task_dir / 'annotations.json', 'w') as f:
                json.dump(annotations, f, indent=2)
            labeled_frame_ids = {
                s.get('frame') for s in annotations.get('shapes', [])
                if s.get('type') == 'polygon'
            }
            print(f"  Annotations saved: {len(annotations.get('shapes', []))} shapes "
                  f"({len(labeled_frame_ids)} labeled frames)")
        except Exception as e:
            print(f"  Error getting annotations (after retries): {e}")
            annotations = {'shapes': []}
            labeled_frame_ids = set()

        # 4. Descargar los frames etiquetados, EN PARALELO
        frames_to_download = [f for f in all_frames if f['id'] in labeled_frame_ids]
        downloaded_frames = client.download_frames_parallel(
            task_id, frames_to_download, frames_dir, max_workers=max_workers,
            desc=f"  Downloading labeled frames"
        )
        print(f"  Labeled frames downloaded: {downloaded_frames}/{len(frames_to_download)} "
              f"(of {len(all_frames)} total in the farm)")

        # 5. Generar manifest.json (incluye los frame ids, no solo los
        # descargados, para poder calcular unlabeled_frames sin tener sus PNG)
        manifest = {
            "task_id": task_id,
            "task_name": name,
            "total_frames": len(all_frames),
            "all_frame_ids": [f['id'] for f in all_frames],
            "downloaded_frames": downloaded_frames,
            "total_annotations": len(annotations.get('shapes', [])),
            "label_id": FARMS.get(name, {}).get('label_id', 8155070)
        }
        with open(task_dir / 'manifest.json', 'w') as f:
            json.dump(manifest, f, indent=2)

        summary[task_id] = manifest
        print(f"  Manifest saved to {task_dir / 'manifest.json'}")

        # 5b. Persistir metadata liviana en /kaggle/output (sobrevive aunque se
        # caiga la sesión; las imágenes en sí no se copian por su tamaño).
        if persist_to_output:
            try:
                import shutil
                dest = kaggle_output_meta / str(task_id)
                dest.mkdir(parents=True, exist_ok=True)
                shutil.copy(task_dir / 'manifest.json', dest / 'manifest.json')
                shutil.copy(task_dir / 'annotations.json', dest / 'annotations.json')
            except Exception as e:
                print(f"  (could not persist metadata to /kaggle/output: {e})")

    return summary

print("download_cvat_data() function defined (labeled-frames-only download, parallel, with metadata persistence).")

In [ ]:
# Descargar datos de las 15 fincas (frames + anotaciones existentes en CVAT)
print("="*60)
print("DOWNLOADING DATA — 15 FARMS")
print("="*60)

client = CvatClient(
    os.environ['CVAT_HOST'],
    os.environ['CVAT_USER'],
    os.environ['CVAT_PASSWORD']
)

all_task_ids = [cfg['task_id'] for cfg in FARMS.values()]
raw_data_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')

download_summary = download_cvat_data(
    all_task_ids,
    raw_data_dir,
    client,
    max_workers=DOWNLOAD_MAX_WORKERS
)

print("\n" + "="*60)
print("DOWNLOAD SUMMARY — 15 FARMS")
print("="*60)
for task_id, info in download_summary.items():
    print(f"  Task {task_id}: {info['task_name']} - {info['downloaded_frames']} frames, {info['total_annotations']} annotations")

---

### 4. Análisis de Etiquetado por Finca

Antes de entrenar o hacer auto-etiquetado, revisamos **frame por frame** qué está
ya anotado en CVAT y qué falta, en cada una de las 15 fincas. Esto define:

- **Datos de entrenamiento**: todo frame que ya tiene al menos un polígono anotado,
  de cualquiera de las 15 fincas.
- **Datos a auto-etiquetar (Sección 8)**: todo frame que no tiene ningún polígono
  anotado todavía, finca por finca.

In [ ]:
def analyze_labeling(raw_data_dir, farms):
    """
    Calcula cobertura de etiquetado por finca usando manifest.json (que ya
    trae la lista completa de frame ids de la task, aunque solo se hayan
    descargado los PNG de los etiquetados) + annotations.json. No depende de
    que los frames sin etiquetar estén en disco.
    """
    raw_data = Path(raw_data_dir)
    result = {}

    for name, cfg in farms.items():
        task_dir = raw_data / str(cfg['task_id'])
        manifest_path = task_dir / 'manifest.json'
        ann_path = task_dir / 'annotations.json'

        if manifest_path.exists():
            manifest_data = json.loads(manifest_path.read_text())
            total_frame_numbers = set(manifest_data.get('all_frame_ids', []))
        else:
            total_frame_numbers = set()

        labeled_frames = set()
        shapes_before = 0
        if ann_path.exists():
            annotations = json.loads(ann_path.read_text())
            for shape in annotations.get('shapes', []):
                if shape.get('type') == 'polygon':
                    shapes_before += 1
                    labeled_frames.add(shape.get('frame'))

        labeled_frames = labeled_frames & total_frame_numbers
        unlabeled_frames = total_frame_numbers - labeled_frames

        result[name] = {
            'task_id': cfg['task_id'],
            'total_frames': len(total_frame_numbers),
            'labeled_frames': sorted(labeled_frames),
            'unlabeled_frames': sorted(unlabeled_frames),
            'shapes_before': shapes_before,
        }

    return result


raw_data_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
LABELING_STATUS = analyze_labeling(raw_data_dir, FARMS)

print("="*92)
print(f"{'Farm':<18}{'Frames':>8}{'Labeled':>10}{'Unlabeled':>11}{'% labeled':>11}{'Shapes before':>16}")
print("="*92)
total_frames = total_labeled = total_unlabeled = total_shapes_before = 0
for name, info in LABELING_STATUS.items():
    n_total = info['total_frames']
    n_labeled = len(info['labeled_frames'])
    n_unlabeled = len(info['unlabeled_frames'])
    pct = (n_labeled / n_total * 100) if n_total else 0.0
    total_frames += n_total
    total_labeled += n_labeled
    total_unlabeled += n_unlabeled
    total_shapes_before += info['shapes_before']
    print(f"{name:<18}{n_total:>8}{n_labeled:>10}{n_unlabeled:>11}{pct:>10.1f}%{info['shapes_before']:>16}")
print("-"*92)
total_pct = (total_labeled / total_frames * 100) if total_frames else 0.0
print(f"{'TOTAL':<18}{total_frames:>8}{total_labeled:>10}{total_unlabeled:>11}{total_pct:>10.1f}%{total_shapes_before:>16}")

# Guardar snapshot
metrics_dir = os.path.join(PROJECT_ROOT, 'outputs', 'metrics')
os.makedirs(metrics_dir, exist_ok=True)
with open(os.path.join(metrics_dir, 'labeling_status_before.json'), 'w') as f:
    json.dump(LABELING_STATUS, f, indent=2)
print(f"\nLabeling status (before) saved to: {metrics_dir}/labeling_status_before.json")

---

## 4. Preparación del Dataset SAM2

Convertir las anotaciones al formato requerido por SAM2 (SA-V format):
- Máscaras binarias → RLE COCO
- Split estratificado 80/20 por finca y densidad
- Generación de `file_list.txt` para train/val

In [ ]:
from pycocotools import mask as coco_mask
import random

# Mapeo de task IDs a nombres de fincas (generado desde FARMS, ya no hardcodeado)
TASK_NAMES = {str(cfg['task_id']): name.lower() for name, cfg in FARMS.items()}
print(f"TASK_NAMES generated for {len(TASK_NAMES)} farms.")

def polygon_to_mask(points, w, h):
    """Convierte polígono flat a máscara binaria h×w."""
    m = np.zeros((h, w), dtype=np.uint8)
    if not points or len(points) < 6:
        return m
    pts = np.array(points, dtype=np.float32).reshape(-1, 2).round().astype(np.int32)
    cv2.fillPoly(m, [pts], 1)
    return m

def mask_to_rle(mask):
    """Convierte máscara binaria a RLE COCO."""
    m = (mask > 0).astype(np.uint8)
    rle = coco_mask.encode(np.asfortranarray(m))
    if isinstance(rle['counts'], bytes):
        rle['counts'] = rle['counts'].decode('ascii')
    return {'size': rle['size'], 'counts': rle['counts']}

def classify_density(n_trees):
    """Clasifica densidad de árboles por frame."""
    if n_trees <= 10:
        return "low"
    elif n_trees <= 25:
        return "medium"
    else:
        return "high"

print("Conversion functions defined.")

In [ ]:
def load_farm_tiles(raw_data_dir):
    """
    Carga tiles de todas las tasks con manifest.json.
    Retorna lista de diccionarios con información de cada tile.
    
    """
    items = []
    skipped_missing_frame = 0
    raw_data = Path(raw_data_dir)

    for task_dir in sorted(raw_data.iterdir()):
        if not task_dir.is_dir():
            continue

        task_id = task_dir.name
        man_path = task_dir / "manifest.json"
        if not man_path.exists():
            continue

        data = json.loads(man_path.read_text())
        prefix = TASK_NAMES.get(task_id, f"task_{task_id}")

        # Cargar anotaciones
        ann_path = task_dir / "annotations.json"
        if not ann_path.exists():
            continue

        annotations = json.loads(ann_path.read_text())
        shapes = annotations.get('shapes', [])

        # Agrupar shapes por frame
        shapes_by_frame = {}
        for shape in shapes:
            frame = shape.get('frame', 0)
            if frame not in shapes_by_frame:
                shapes_by_frame[frame] = []
            shapes_by_frame[frame].append(shape)

        # Crear items para cada frame con anotaciones
        for frame_num, frame_shapes in shapes_by_frame.items():
            if not frame_shapes:
                continue

            # Extraer puntos de polígonos
            polygons = []
            for shape in frame_shapes:
                if shape.get('type') == 'polygon':
                    polygons.append(shape.get('points', []))

            if not polygons:
                continue

            # Verificar que el frame existe
            frame_path = task_dir / "frames" / f"frame_{frame_num:05d}.png"
            if not frame_path.exists():
                skipped_missing_frame += 1
                continue

            items.append({
                "video_id": f"{prefix}_{frame_num:05d}",
                "task_id": int(task_id),
                "frame": frame_num,
                "polygons": polygons,
                "raw_frame": frame_path,
                "farm": prefix
            })

    if skipped_missing_frame:
        print(f"WARNING: {skipped_missing_frame} labeled frame(s) skipped because their raw PNG.")

    return items

print("load_farm_tiles() function defined.")

In [ ]:
def prepare_sam2_dataset(raw_data_dir, output_dir, split_ratio=0.8):
    """
    Prepara el dataset en formato SAM2 SA-V.

    1. Carga tiles de todas las fincas de entrenamiento
    2. Convierte polígonos a máscaras RLE COCO
    3. Divide en train/val (80/20 estratificado)
    4. Genera estructura de carpetas y file_list.txt
    """
    output_root = Path(output_dir)

    print("Loading training farm tiles...")
    items = load_farm_tiles(raw_data_dir)
    print(f"Total tiles found: {len(items)}")

    if not items:
        print("ERROR: No training tiles found.")
        return 0, 0

    # Agrupar por finca para split estratificado
    by_farm = {}
    for item in items:
        farm = item['farm']
        if farm not in by_farm:
            by_farm[farm] = []
        by_farm[farm].append(item)

    print(f"Distribution by farm:")
    for farm, tiles in by_farm.items():
        print(f"  {farm}: {len(tiles)} tiles")

    # Dividir en train/val estratificado
    train_items = []
    val_items = []

    random.seed(42)  # Para reproducibilidad

    for farm, tiles in by_farm.items():
        random.shuffle(tiles)
        n_train = int(len(tiles) * split_ratio)
        train_items.extend(tiles[:n_train])
        val_items.extend(tiles[n_train:])

    print(f"\nSplit:")
    print(f"  Train: {len(train_items)} tiles")
    print(f"  Val: {len(val_items)} tiles")

    # Crear estructura de carpetas
    for split_name, split_items in [('train', train_items), ('val', val_items)]:
        split_dir = output_root / split_name
        img_dir = split_dir / 'img' / 'videos'
        gt_dir = split_dir / 'gt' / 'videos'
        img_dir.mkdir(parents=True, exist_ok=True)
        gt_dir.mkdir(parents=True, exist_ok=True)

        video_ids = []

        for item in tqdm(split_items, desc=f"Processing {split_name}"):
            video_id = item['video_id']
            video_ids.append(video_id)

            # Crear carpeta para este video
            vid_img_dir = img_dir / video_id
            vid_img_dir.mkdir(parents=True, exist_ok=True)

            # Copiar imagen original
            img_src = item['raw_frame']
            img_dst = vid_img_dir / '00000.jpg'
            if not img_dst.exists():
                img_bgr = cv2.imread(str(img_src))
                if img_bgr is not None:
                    cv2.imwrite(str(img_dst), img_bgr)

            # Leer imagen para obtener dimensiones
            img_bgr = cv2.imread(str(img_dst))
            if img_bgr is None:
                continue
            h, w = img_bgr.shape[:2]

            # Convertir polígonos a máscaras RLE
            masklets = []
            for poly_points in item['polygons']:
                mask = polygon_to_mask(poly_points, w, h)
                if mask.sum() > 0:
                    rle = mask_to_rle(mask)
                    masklets.append(rle)

            if not masklets:
                continue

            # Guardar en formato SAM2 SA-V
            gt_data = {
                "video_ids": [video_id],
                "videos": {
                    video_id: {
                        "id": video_id,
                        "geo": None,
                        "refframe": 0,
                        "size": [h, w],
                        "frames": {
                            "0": {
                                "frame_index": 0,
                                "is_key_frame": True,
                                "point_coords": [],
                                "labels": [],
                                "bounding_boxes": [],
                                "masklet": [masklets]
                            }
                        }
                    }
                }
            }

            gt_path = gt_dir / f"{video_id}_manual.json"
            with open(gt_path, 'w') as f:
                json.dump(gt_data, f)

        # Generar file_list.txt
        file_list_path = split_dir / 'file_list.txt'
        with open(file_list_path, 'w') as f:
            f.write('\n'.join(sorted(video_ids)))

        print(f"{split_name}: {len(video_ids)} videos saved")

    print(f"\nDataset prepared at: {output_root}")
    return len(train_items), len(val_items)

print("prepare_sam2_dataset() function defined.")

In [ ]:
# Preparar dataset SAM2
print("="*60)
print("SAM2 DATASET PREPARATION")
print("="*60)

raw_data_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
processed_data_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')

n_train, n_val = prepare_sam2_dataset(
    raw_data_dir,
    processed_data_dir,
    SPLIT_RATIO
)

print(f"\nDataset ready for training:")
print(f"  Train: {n_train} tiles")
print(f"  Val: {n_val} tiles")

---

## 5. Entrenamiento del Clasificador de Copas

El clasificador distingue entre **crop_tree** y **background** 

`background = imagen - anotaciones`

Utiliza un RandomForest sobre features de:
- **Color HSV** (distribución de tono, saturación, brillo)
- **Forma** (circularidad, excentricidad)
- **Textura** (varianza del Laplaciano)
- **Espacial** (área relativa, posición)

El proceso incluye:
1. Extracción de crops positivos (crop_tree) de frames ya etiquetados
2. Generación automática de crops negativos (background) por traslación
3. Generación de contact sheets para inspección visual — con borde VERDE
   para crop_tree y ROJO para background, más sheets de ejemplo por clase
   (`crop_tree_examples.png`, `background_examples.png`) que se muestran
   directamente en el notebook justo después de entrenar
4. Entrenamiento con cross-validation + hold-out (precisión/recall/F1)
5. Guardado del modelo y métricas


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report,
                             precision_recall_curve, average_precision_score)
import pickle

TILE_SIZE = 96  # Tamaño de crops para el clasificador

def extract_crops(rgb_image, masks, frame_no=0, pad_ratio=0.15):
    """
    Extrae crops de imágenes basándose en máscaras de segmentación.
    Cada crop se redimensiona a TILE_SIZE x TILE_SIZE.
    """
    h, w = rgb_image.shape[:2]
    crops = []

    for i, mask in enumerate(masks):
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue

        # Bounding box con padding
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        px, py = int((x1 - x0) * pad_ratio), int((y1 - y0) * pad_ratio)
        x0, x1 = max(0, x0 - px), min(w, x1 + px)
        y0, y1 = max(0, y0 - py), min(h, y1 + py)

        # Extraer crop
        crop_img = rgb_image[y0:y1, x0:x1].copy()
        crop_mask = mask[y0:y1, x0:x1].astype(bool)

        # Redimensionar a TILE_SIZE
        scale = TILE_SIZE / max(crop_img.shape[:2])
        nw = max(1, int(round(crop_img.shape[1] * scale)))
        nh = max(1, int(round(crop_img.shape[0] * scale)))
        crop_img = cv2.resize(crop_img, (nw, nh), interpolation=cv2.INTER_AREA)
        crop_mask = cv2.resize(crop_mask.astype(np.uint8), (nw, nh),
                            interpolation=cv2.INTER_NEAREST) > 0

        # Colocar en canvas de tamaño fijo
        canvas_img = np.zeros((TILE_SIZE, TILE_SIZE, 3), np.uint8)
        canvas_mask = np.zeros((TILE_SIZE, TILE_SIZE), bool)
        oy, ox = (TILE_SIZE - nh) // 2, (TILE_SIZE - nw) // 2
        canvas_img[oy:oy + nh, ox:ox + nw] = crop_img
        canvas_mask[oy:oy + nh, ox:ox + nw] = crop_mask

        crops.append({
            "frame": frame_no,
            "mask_index": i,
            "bbox": [x0, y0, x1, y1],
            "image": canvas_img,
            "mask": canvas_mask,
        })

    return crops

print("extract_crops() function defined.")

def generate_background_by_translation(union_mask, poly_points, h, w, rng,
                                  max_attempts=10, max_overlap_fraction=0.02):
    """
    Genera una máscara de "background" (fondo) a partir de un polígono
    crop_tree ya anotado, trasladándolo a una posición aleatoria de la
    MISMA imagen donde no se solape con ningún crop_tree anotado.
    
    """
    pts = np.array(poly_points, dtype=np.float32).reshape(-1, 2)
    x0, y0 = pts[:, 0].min(), pts[:, 1].min()
    x1, y1 = pts[:, 0].max(), pts[:, 1].max()
    poly_w, poly_h = x1 - x0, y1 - y0

    if poly_w >= w or poly_h >= h:
        return None  # el polígono no cabe en ningún desplazamiento

    for _ in range(max_attempts):
        nx0 = rng.uniform(0, w - poly_w)
        ny0 = rng.uniform(0, h - poly_h)
        dx, dy = nx0 - x0, ny0 - y0

        new_pts = pts + np.array([dx, dy], dtype=np.float32)
        mask_bg = polygon_to_mask(new_pts.flatten().tolist(), w, h)

        area = mask_bg.sum()
        if area == 0:
            continue

        overlap = np.logical_and(mask_bg > 0, union_mask > 0).sum()
        if overlap / area <= max_overlap_fraction:
            return mask_bg

    return None


print("generate_background_by_translation() function defined.")

In [ ]:
def extract_features(image, mask):
    """
    Extrae vector de features de 48 dimensiones de un crop.
    Incluye: HSV (40 dims), color RGB (3), forma (4), textura (1).
    """
    _BINS_H, _BINS_S, _BINS_V = 16, 12, 12
    img = image.astype(np.uint8)
    m = mask.astype(bool)

    if m.sum() == 0:
        return np.zeros(48, dtype=np.float32)

    # Histogramas HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h = np.histogram(hsv[..., 0][m], bins=_BINS_H, range=(0, 180),
                     density=True)[0]
    s = np.histogram(hsv[..., 1][m], bins=_BINS_S, range=(0, 256),
                     density=True)[0]
    v = np.histogram(hsv[..., 2][m], bins=_BINS_V, range=(0, 256),
                     density=True)[0]

    # Color promedio RGB
    color_mean = img[m].reshape(-1, 3).mean(axis=0) / 255.0

    # Features de forma
    area_frac = m.mean()
    cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt = max(cnts, key=cv2.contourArea)
        peri = cv2.arcLength(cnt, True)
        area = cv2.contourArea(cnt)
        circularity = 4 * np.pi * area / (peri ** 2) if peri > 0 else 0.0
        (x, y), (MA, ma), angle = cv2.fitEllipse(cnt) if len(cnt) >= 5 \
            else ((0, 0), (1, 1), 0)
        eccentricity = np.sqrt(max(0.0, 1 - (min(MA, ma) / max(MA, ma)) ** 2)) \
            if max(MA, ma) > 0 else 0.0
    else:
        circularity = eccentricity = 0.0

    # Textura (varianza del Laplaciano)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    texture = float(lap[m].var())

    # Concatenar todas las features
    features = np.concatenate([
        h, s, v,
        color_mean,
        [area_frac, circularity, eccentricity, float(m.sum())],
        [texture],
    ]).astype(np.float32)

    return features

print("extract_features() function defined.")
print(f"Feature dimensions: {extract_features(
    np.zeros((96, 96, 3), dtype=np.uint8),
    np.ones((96, 96), dtype=bool)
).shape[0]}")

In [ ]:
def generate_contact_sheet(crops, start_index=0, labels=None):
    """
    Genera contact sheet para inspección visual de crops.

    labels: lista opcional (mismo largo que crops) con 1=crop_tree, 0=background.
    Si se da, cada crop se dibuja con un borde de color para poder distinguirlos
    de un vistazo: VERDE = crop_tree, ROJO = background (fondo auto-generado).
    """
    SHEET_COLS = 8
    n = len(crops)
    rows = (n + SHEET_COLS - 1) // SHEET_COLS
    sheet = np.full((rows * TILE_SIZE, SHEET_COLS * TILE_SIZE, 3), 255, np.uint8)

    for k, c in enumerate(crops):
        r, col = divmod(k, SHEET_COLS)
        y0, x0 = r * TILE_SIZE, col * TILE_SIZE
        sheet[y0:y0 + TILE_SIZE, x0:x0 + TILE_SIZE] = c["image"]

        if labels is not None:
            is_crop_tree = labels[k] == 1
            border_color = (0, 200, 0) if is_crop_tree else (0, 0, 230)  # RGB: verde / rojo
            thickness = 3
            label_text = "CT" if is_crop_tree else "BG"
        else:
            border_color = (0, 0, 0)
            thickness = 1
            label_text = str(start_index + k)

        cv2.rectangle(sheet, (x0, y0), (x0 + TILE_SIZE - 1, y0 + TILE_SIZE - 1),
                      border_color, thickness)
        (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(sheet, (x0 + 2, y0 + 2),
                      (x0 + 4 + tw, y0 + 6 + th), (255, 255, 255), -1)
        cv2.putText(sheet, label_text, (x0 + 4, y0 + 16),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 200), 1,
                    cv2.LINE_AA)

    return sheet

print("generate_contact_sheet() function defined (color-coded by class: green=crop_tree, red=background).")

In [ ]:
def train_classifier(raw_data_dir, output_dir, n_trees=300, max_tiles=200):
    """
    Entrena el clasificador de copas (crop_tree vs background).

    Proceso:
    1. Extrae crops POSITIVOS (crop_tree) de frames ya etiquetados en CVAT
    2. Genera crops NEGATIVOS (background) trasladando cada polígono positivo
       a una zona sin solape del mismo frame (background = imagen - anotaciones,
       solo en frames ya etiquetados; nunca en frames pendientes de etiquetar)
    3. Genera contact sheets para inspección
    4. Entrena RandomForest con cross-validation + hold-out
    5. Guarda modelo y métricas
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    print("Loading tiles for crop extraction...")
    items = load_farm_tiles(raw_data_dir)
    print(f"Total tiles: {len(items)}")

    # Tomar muestra si hay muchos tiles (max_tiles=None -> usar todos)
    if max_tiles is not None and len(items) > max_tiles:
        random.seed(42)
        items = random.sample(items, max_tiles)
        print(f"Sample taken: {max_tiles} tiles")
    else:
        print(f"Using all available tiles: {len(items)}")

    # Extraer crops de cada tile: positivos (crop_tree, anotados) + negativos (background, generados automáticamente).

    BG_SAMPLES_PER_POSITIVE = 1
    rng_bg = random.Random(1234)

    all_crops = []
    all_labels = []
    n_bg_failed = 0

    for item in tqdm(items, desc="Extracting crops"):
        img_bgr = cv2.imread(str(item['raw_frame']))
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Convertir polígonos a máscaras (positivos = crop_tree)
        masks = []
        for poly_points in item['polygons']:
            m = polygon_to_mask(poly_points, w, h)
            if m.sum() > 0:
                masks.append(m)

        if not masks:
            continue

        # Unión de todos los crop_tree del frame (para no solaparnos al generar fondo)
        union_mask = np.zeros((h, w), dtype=np.uint8)
        for m in masks:
            union_mask = np.maximum(union_mask, m)

        # Crops positivos (crop_tree)
        positive_crops = extract_crops(img_rgb, masks, frame_no=item['frame'])
        for c in positive_crops:
            c['farm'] = item['farm']
        all_crops.extend(positive_crops)
        all_labels.extend([1] * len(positive_crops))

        # Crops negativos (background), por traslación de cada polígono positivo
        bg_masks = []
        for poly_points in item['polygons']:
            for _ in range(BG_SAMPLES_PER_POSITIVE):
                m_bg = generate_background_by_translation(union_mask, poly_points, h, w, rng=rng_bg)
                if m_bg is not None:
                    bg_masks.append(m_bg)
                else:
                    n_bg_failed += 1

        if bg_masks:
            background_crops = extract_crops(img_rgb, bg_masks, frame_no=item['frame'])
            for c in background_crops:
                c['farm'] = item['farm']
            all_crops.extend(background_crops)
            all_labels.extend([0] * len(background_crops))

    n_pos = int(sum(all_labels))
    n_bg = len(all_labels) - n_pos
    print(f"Total crops extracted: {len(all_crops)} (crop_tree: {n_pos}, background: {n_bg})")
    if n_bg_failed:
        print(f"  (could not generate background for {n_bg_failed} instances: "
              f"frame too full of crop_tree to find free space)")

    if len(all_crops) == 0:
        print("ERROR: Could not extract any crops.")
        return

    # Generar contact sheets (todos los crops, en orden, con color por clase:
    # borde VERDE = crop_tree, borde ROJO = background auto-generado)
    n_sheets = (len(all_crops) + 79) // 80
    for s in range(n_sheets):
        batch = all_crops[s*80:(s+1)*80]
        batch_labels = all_labels[s*80:(s+1)*80]
        sheet = generate_contact_sheet(batch, start_index=s*80, labels=batch_labels)
        cv2.imwrite(str(output_path / f"sheet_{s:02d}.png"),
                    cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    print(f"Contact sheets generated: {n_sheets} (at {output_path}/sheet_XX.png)")

    # Sheets de ejemplo, separados por clase 
    crop_tree_indices = [i for i, l in enumerate(all_labels) if l == 1]
    background_indices = [i for i, l in enumerate(all_labels) if l == 0]

    EXAMPLES_PER_FARM = 3
    examples_rng = random.Random(2024)

    def _save_examples(indices, file_name, per_farm=EXAMPLES_PER_FARM):
        
        if not indices:
            return 0, 0
        by_farm = {}
        for i in indices:
            by_farm.setdefault(all_crops[i].get('farm', 'unknown'), []).append(i)

        sample = []
        for farm_indices in by_farm.values():
            k = min(per_farm, len(farm_indices))
            sample.extend(examples_rng.sample(farm_indices, k))

        sample_crops = [all_crops[i] for i in sample]
        sample_labels = [all_labels[i] for i in sample]
        sheet = generate_contact_sheet(sample_crops, labels=sample_labels)
        cv2.imwrite(str(output_path / file_name), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
        return len(sample), len(by_farm)

    n_ct, farms_ct = _save_examples(crop_tree_indices, "crop_tree_examples.png")
    n_bg, farms_bg = _save_examples(background_indices, "background_examples.png")
    print(f"Examples per class saved: {output_path}/crop_tree_examples.png "
          f"({n_ct} crops across {farms_ct} farms), "
          f"{output_path}/background_examples.png ({n_bg} crops across {farms_bg} farms)")

    # Guardar crops para uso futuro
    images = np.stack([c["image"] for c in all_crops])
    masks_arr = np.stack([c["mask"] for c in all_crops])
    manifest = [{"frame": c["frame"], "mask_index": c["mask_index"],
                 "bbox": c["bbox"], "label": lbl, "farm": c.get("farm")}
                for c, lbl in zip(all_crops, all_labels)]

    np.savez_compressed(output_path / "crops.npz", images=images, masks=masks_arr)
    (output_path / "crops_manifest.json").write_text(json.dumps(manifest, indent=2))

    labels = np.array(all_labels, dtype=int)  # 1 = crop_tree, 0 = background (auto-generado)

    # Extraer features
    print("Extracting features...")
    X = np.stack([extract_features(images[i], masks_arr[i]) for i in range(len(all_crops))])
    y = labels

    print(f"Dataset: {len(y)} samples ({(y==0).sum()} background, {(y==1).sum()} crop_tree)")
    print(f"Features shape: {X.shape}")

    has_real_labels = len(set(y.tolist())) > 1
    if not has_real_labels:
        print("\n" + "!"*70)
        print("IMPORTANT NOTICE:")
        print("  Could not generate ANY automatic background (frames too dense")
        print("  or too few labeled frames). All crops ended up as")
        print("  'crop_tree', so this classifier's metrics")
        print("  (accuracy, CV, etc.) are NOT valid for the article.")
        print("!"*70 + "\n")

    scores = None
    test_metrics = None

    if has_real_labels and len(y) >= 10:
        # Cross-validation (estimación de generalización)
        clf_cv = RandomForestClassifier(n_estimators=n_trees, random_state=42,
                                        class_weight="balanced", n_jobs=-1)
        cv_folds = min(5, min(np.bincount(y)))
        cv_folds = max(cv_folds, 2)
        scores = cross_val_score(clf_cv, X, y,
                                 cv=StratifiedKFold(cv_folds, shuffle=True, random_state=42))
        print(f"CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

        # Hold-out 80/20 para métricas de precisión/recall/F1
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        clf_eval = RandomForestClassifier(n_estimators=n_trees, random_state=42,
                                          class_weight="balanced", n_jobs=-1)
        clf_eval.fit(X_tr, y_tr)
        y_pred = clf_eval.predict(X_te)

        test_accuracy = accuracy_score(y_te, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_te, y_pred, average="binary", zero_division=0
        )
        cm = confusion_matrix(y_te, y_pred).tolist()

        print(f"\nHold-out test (20%): {len(y_te)} samples")
        print(f"  Accuracy:  {test_accuracy:.3f}")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall:    {recall:.3f}")
        print(f"  F1-score:  {f1:.3f}")
        print(f"  Confusion matrix (rows=actual, cols=predicted): {cm}")
        print("\n" + classification_report(y_te, y_pred, target_names=["background", "crop_tree"], zero_division=0))

        # Curva Precision-Recall (barre el umbral de decisión en vez de fijarlo
        # en 0.5 como arriba) + Average Precision (AP, área bajo la curva).
        y_scores = clf_eval.predict_proba(X_te)[:, 1]
        pr_precision, pr_recall, _ = precision_recall_curve(y_te, y_scores)
        ap_score = average_precision_score(y_te, y_scores)
        print(f"  Average Precision (AP): {ap_score:.3f}")

        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(5.5, 5))
            plt.plot(pr_recall, pr_precision, color="steelblue", linewidth=2,
                     label=f"RandomForest (AP={ap_score:.3f})")
            plt.axhline(y=(y_te == 1).mean(), color="gray", linestyle="--",
                        label="Random classifier")
            plt.xlabel("Recall"); plt.ylabel("Precision")
            plt.xlim(0, 1); plt.ylim(0, 1.05)
            plt.title("Precision-Recall Curve — crop_tree vs background")
            plt.legend(); plt.grid(alpha=0.3)
            pr_curve_path = output_path / "precision_recall_curve.png"
            plt.tight_layout()
            plt.savefig(pr_curve_path, dpi=150)
            plt.show()
            print(f"  PR curve saved to: {pr_curve_path}")
        except Exception as e:
            print(f"  Could not generate PR curve: {e}")

        test_metrics = {
            "test_samples": int(len(y_te)),
            "test_accuracy": float(test_accuracy),
            "test_precision": float(precision),
            "test_recall": float(recall),
            "test_f1": float(f1),
            "confusion_matrix": cm,
            "average_precision": float(ap_score),
        }

    # Entrenar modelo final con los datos disponibles
    print("\nTraining final model...")
    clf = RandomForestClassifier(n_estimators=n_trees, random_state=42,
                                 class_weight="balanced" if has_real_labels else None, n_jobs=-1)
    clf.fit(X, y)
    train_acc = clf.score(X, y)
    print(f"Train accuracy: {train_acc:.3f}" + ("" if has_real_labels else "  (not informative: single class)"))

    # Feature importance
    feat_names = (
        [f"H{i}" for i in range(16)] +
        [f"S{i}" for i in range(12)] +
        [f"V{i}" for i in range(12)] +
        ["R", "G", "B"] +
        ["area_frac", "circularity", "eccentricity", "area"] +
        ["texture"]
    )  # 48 nombres, en el mismo orden que extract_features() concatena el vector
    importances = clf.feature_importances_
    top_idx = np.argsort(importances)[::-1][:10]
    print("\nTop 10 features:")
    for idx in top_idx:
        print(f"  {feat_names[idx]:12s} {importances[idx]:.4f}")

    # Guardar modelo
    model_path = output_path / "crown_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(clf, f)
    print(f"\nModel saved: {model_path}")

    # Guardar métricas (para el artículo)
    metrics = {
        "has_real_labels": bool(has_real_labels),
        "train_samples": int(len(y)),
        "train_background": int((y == 0).sum()),
        "train_crop_tree": int((y == 1).sum()),
        "train_accuracy": float(train_acc),
        "cv_accuracy": float(scores.mean()) if scores is not None else None,
        "cv_std": float(scores.std()) if scores is not None else None,
        "feature_dim": int(X.shape[1]),
    }
    if test_metrics:
        metrics.update(test_metrics)
    (output_path / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"Metrics saved: {output_path / 'metrics.json'}")

    return clf

print("train_classifier() function defined.")

In [ ]:
# Entrenar clasificador de copas
print("="*60)
print("CROWN CLASSIFIER TRAINING")
print("="*60)

classifier_dir = os.path.join(PROJECT_ROOT, 'outputs', 'classifier')

classifier = train_classifier(
    raw_data_dir,
    classifier_dir,
    EPOCHS_CLF,
    max_tiles=MAX_TILES_CLASSIFIER
)

print("\n" + "="*60)
print("CLASSIFIER TRAINED")
print("="*60)

In [ ]:
# Ver ejemplos de crop_tree vs. background
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

examples_paths = {
    "crop_tree (green)": os.path.join(classifier_dir, "crop_tree_examples.png"),
    "background (red, auto-generated)": os.path.join(classifier_dir, "background_examples.png"),
}

for title, path in examples_paths.items():
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.title(f"Examples: {title}")
        plt.axis("off")
        plt.show()
    else:
        print(f"{path} was not generated (no crops of that class?)")

print(f"\nFull sheets (all crops, in order): {classifier_dir}/sheet_XX.png")
print("  GREEN border = crop_tree | RED border = auto-generated background")

---

## 6. Fine-tuning de SAM2

Entrenar el **mask decoder** y **prompt encoder** de SAM2 sobre el
dataset de árboles frutales. El **encoder de imágenes permanece congelado**
para preservar las features pre-entrenadas.

Configuración de entrenamiento:
- **Modelo base**: sam2.1_hiera_large
- **Encoder**: Congelado (vision_lr=0)
- **Decoder + Prompt Encoder**: Entrenables (lr=1e-5)
- **Dice Loss**: Loss
- **Optimizer**: AdamW (weight_decay=0.01)
- **Epochs**: Configurable (default: 30)

Además del loss de entrenamiento, en cada epoch se evalúa el modelo sobre el
set de **validación** con 4 métricas a nivel de píxel — IoU, Dice, Precision y
Recall —, se guarda el checkpoint del **mejor epoch** (`sam2_finetuned_best.pt`,
no solo el último) y al final se genera:
- `outputs/metrics/sam2_training_history.json` — historial epoch a epoch
- `outputs/metrics/sam2_eval_final.json` — métricas finales (val IoU/Dice/Precision/Recall)
- `outputs/metrics/sam2_training_curves.png` — gráfica de 3 paneles
  **(1)** curva de pérdida (error), **(2)** IoU (métrica principal de
  segmentación) y **(3)** Precision/Recall (para saber si el modelo sobre-segmenta
  o deja árboles sin cubrir)
- Un diagnóstico automático de si `EPOCHS_SAM2` fue suficiente

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset

class TreeDataset(Dataset):
    """
    Dataset para cargar tiles de árboles en formato SAM2 SA-V.
    Retorna imagen RGB y máscara unión de todos los árboles del tile.
    """

    def __init__(self, root, split):
        self.root = Path(root) / split
        file_list_path = self.root / 'file_list.txt'
        if not file_list_path.exists():
            raise FileNotFoundError(f"{file_list_path} not found")
        self.ids = file_list_path.read_text().strip().split('\n')

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        vid = self.ids[idx]

        # Cargar imagen
        img_path = self.root / 'img' / 'videos' / vid / '00000.jpg'
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

        # Cargar máscaras GT
        gt_path = self.root / 'gt' / 'videos' / f'{vid}_manual.json'
        gt = json.loads(gt_path.read_text())

        vid2 = gt['video_ids'][0]
        h, w = gt['videos'][vid2]['size']

        # Crear máscara unión de todas las instancias
        union = np.zeros((h, w), np.uint8)
        for frame_data in gt['videos'][vid2]['frames'].values():
            for masklet_group in frame_data.get('masklet', []):
                for rle in masklet_group:
                    rr = dict(rle)
                    if isinstance(rr['counts'], str):
                        rr['counts'] = rr['counts'].encode('ascii')
                    decoded = coco_mask.decode(rr)
                    union = np.maximum(union, decoded)

        return img, union, vid

print("TreeDataset class defined.")

In [ ]:
# Descargar checkpoint base de SAM2 (tamaño según SAM2_MODEL_SIZE)
SAM2_VARIANTS = {
    "large":     {"file": "sam2.1_hiera_large.pt",     "config": "configs/sam2.1/sam2.1_hiera_l.yaml",  "min_bytes": 850e6},
    "base_plus": {"file": "sam2.1_hiera_base_plus.pt", "config": "configs/sam2.1/sam2.1_hiera_b+.yaml", "min_bytes": 300e6},
    "small":     {"file": "sam2.1_hiera_small.pt",     "config": "configs/sam2.1/sam2.1_hiera_s.yaml",  "min_bytes": 150e6},
    "tiny":      {"file": "sam2.1_hiera_tiny.pt",      "config": "configs/sam2.1/sam2.1_hiera_t.yaml",  "min_bytes": 100e6},
}
variant = SAM2_VARIANTS[SAM2_MODEL_SIZE]

CKPT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
BASE_CKPT = os.path.join(CKPT_DIR, variant["file"])
SAM2_CONFIG = variant["config"]

if not os.path.exists(BASE_CKPT) or os.path.getsize(BASE_CKPT) < variant["min_bytes"]:
    print(f"Downloading SAM2 base checkpoint ({SAM2_MODEL_SIZE})...")
    url_ckpt = f"https://dl.fbaipublicfiles.com/segment_anything_2/092824/{variant['file']}"
    !wget -q {url_ckpt} -O {BASE_CKPT}
    print("Download complete.")
else:
    print(f"Base checkpoint already exists: {os.path.getsize(BASE_CKPT)/1e6:.0f} MB")

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Construir modelo SAM2
print("Building SAM2 model...")
sam2_model = build_sam2(SAM2_CONFIG, BASE_CKPT, device='cuda')
predictor = SAM2ImagePredictor(sam2_model)

# Congelar encoder (no entrenable)
for param in sam2_model.image_encoder.parameters():
    param.requires_grad = False

# Parámetros entrenables: sampler + mask decoder
trainable_params = list(sam2_model.sam_prompt_encoder.parameters()) + \
                     list(sam2_model.sam_mask_decoder.parameters())

optimizer = torch.optim.AdamW(trainable_params, lr=1e-5, weight_decay=0.01)

print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")
print("Encoder frozen. Decoder + PromptEncoder trainable.")

In [ ]:
def dice_loss(pred_logits, gt):
    """Calcula el Loss entre predicción y ground truth."""
    pred = torch.sigmoid(pred_logits).clamp(1e-6, 1-1e-6)
    gt = gt.float()
    inter = (pred * gt).sum()
    denom = pred.sum() + gt.sum() + 1e-6
    return 1 - (2 * inter + 1) / denom

def forward_tile(predictor, img, union_mask):
    """
    Ejecuta forward pass de SAM2 en un tile.
    Encoder congelado + decoder entrenable.
    """
    # Encoder congelado + features
    predictor.reset_predictor()
    predictor.set_image(img)
    img_hw = predictor._orig_hw[0]
    feats = predictor._features

    # Box prompt = bbox de la unión GT
    ys, xs = np.where(union_mask > 0)
    if len(xs) == 0:
        return None

    x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
    box = torch.as_tensor([[x0, y0, x1, y1]], dtype=torch.float32, device='cuda')
    box_coords = box.reshape(-1, 2, 2)
    box_labels = torch.tensor([[2, 3]], dtype=torch.int, device='cuda').repeat(box.size(0), 1)
    points = (box_coords, box_labels)

    # Prompt encoder
    sparse_emb, dense_emb = predictor.model.sam_prompt_encoder(
        points=points, boxes=None, masks=None
    )

    # Mask decoder
    high_res = [f[0].unsqueeze(0) for f in feats['high_res_feats']]
    uses_hr = predictor.model.use_high_res_features_in_sam
    low_res, iou, _, _ = predictor.model.sam_mask_decoder(
        image_embeddings=feats['image_embed'][0].unsqueeze(0),
        image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_emb,
        dense_prompt_embeddings=dense_emb,
        multimask_output=False,
        repeat_image=False,
        high_res_features=high_res if uses_hr else None,
    )

    # Interpolar a tamaño original
    pred_logits = F.interpolate(
        low_res, size=img_hw, mode='bilinear', align_corners=False
    )[0, 0]

    # Calcular loss
    gt = torch.as_tensor(union_mask > 0, dtype=torch.float32, device='cuda')
    return dice_loss(pred_logits, gt)

print("SAM2 training functions defined.")

def calculate_iou_dice(pred_mask, gt_mask):
    """IoU y Dice entre una máscara binaria predicha y el ground truth."""
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    iou = float(inter / union) if union > 0 else 1.0
    denom = pred.sum() + gt.sum()
    dice = float(2 * inter / denom) if denom > 0 else 1.0
    return iou, dice


def calculate_pixel_precision_recall(pred_mask, gt_mask):
    """
    Precisión y recall a nivel de píxel entre la máscara predicha y el GT.

    Útil para diagnosticar el tipo de error de segmentación (IoU/Dice  mezclan ambos en un solo número):
      - Precisión baja  -> el modelo sobre-segmenta (mancha zonas que no son árbol)
      - Recall bajo     -> el modelo se queda corto (deja árboles sin cubrir)
    """
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()

    if tp + fp == 0:
        precision = 1.0 if gt.sum() == 0 else 0.0
    else:
        precision = float(tp / (tp + fp))

    if tp + fn == 0:
        recall = 1.0 if pred.sum() == 0 else 0.0
    else:
        recall = float(tp / (tp + fn))

    return precision, recall


def evaluate_val(predictor, sam2_model, dataset, max_samples=None, seed=123):
    """
    Evalúa el modelo (sin backprop) sobre un dataset (train o val):  devuelve loss promedio, IoU promedio y Dice promedio.
    max_samples: si se da, evalúa sobre una submuestra aleatoria (más rápido para correr una vez por epoch); usar None para la evaluación final completa.
    """
    sam2_model.eval()
    ids = list(range(len(dataset)))
    if max_samples is not None and max_samples < len(ids):
        rnd = random.Random(seed)
        ids = rnd.sample(ids, max_samples)

    losses, ious, dices, precisions, recalls = [], [], [], [], []
    with torch.no_grad():
        for idx in ids:
            img, union, vid = dataset[idx]
            if union.sum() == 0:
                continue

            predictor.reset_predictor()
            predictor.set_image(img)
            img_hw = predictor._orig_hw[0]
            feats = predictor._features

            ys, xs = np.where(union > 0)
            if len(xs) == 0:
                continue
            x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
            box = torch.as_tensor([[x0, y0, x1, y1]], dtype=torch.float32, device='cuda')
            box_coords = box.reshape(-1, 2, 2)
            box_labels = torch.tensor([[2, 3]], dtype=torch.int, device='cuda').repeat(box.size(0), 1)
            points = (box_coords, box_labels)

            sparse_emb, dense_emb = predictor.model.sam_prompt_encoder(
                points=points, boxes=None, masks=None
            )
            high_res = [f[0].unsqueeze(0) for f in feats['high_res_feats']]
            uses_hr = predictor.model.use_high_res_features_in_sam
            low_res, _, _, _ = predictor.model.sam_mask_decoder(
                image_embeddings=feats['image_embed'][0].unsqueeze(0),
                image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_emb,
                dense_prompt_embeddings=dense_emb,
                multimask_output=False,
                repeat_image=False,
                high_res_features=high_res if uses_hr else None,
            )
            pred_logits = F.interpolate(
                low_res, size=img_hw, mode='bilinear', align_corners=False
            )[0, 0]

            gt_t = torch.as_tensor(union > 0, dtype=torch.float32, device='cuda')
            loss = dice_loss(pred_logits, gt_t)
            losses.append(loss.item())

            pred_mask = (torch.sigmoid(pred_logits) > 0.5).cpu().numpy()
            iou, dice = calculate_iou_dice(pred_mask, union > 0)
            precision, recall = calculate_pixel_precision_recall(pred_mask, union > 0)
            ious.append(iou)
            dices.append(dice)
            precisions.append(precision)
            recalls.append(recall)

    sam2_model.train()
    if not losses:
        return {"loss": None, "iou": None, "dice": None, "precision": None, "recall": None, "n_samples": 0}
    return {
        "loss": float(np.mean(losses)),
        "iou": float(np.mean(ious)),
        "dice": float(np.mean(dices)),
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "n_samples": len(losses),
    }

print("Evaluation functions (pixel-level IoU/Dice/Precision/Recall) defined.")

In [ ]:
# Preparar datasets
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data', 'processed')

train_ds = TreeDataset(DATA_ROOT, 'train')
val_ds = TreeDataset(DATA_ROOT, 'val')
print(f"Train: {len(train_ds)} tiles")
print(f"Val: {len(val_ds)} tiles")

In [ ]:
# Entrenamiento de SAM2 (con validación de IoU/Dice por epoch)
OUTPUT_CKPT = os.path.join(CKPT_DIR, 'sam2_finetuned.pt')
OUTPUT_CKPT_BEST = os.path.join(CKPT_DIR, 'sam2_finetuned_best.pt')
KAGGLE_OUTPUT = '/kaggle/output/sam2_finetuned.pt'
KAGGLE_OUTPUT_BEST = '/kaggle/output/sam2_finetuned_best.pt'

metrics_dir = os.path.join(PROJECT_ROOT, 'outputs', 'metrics')
os.makedirs(metrics_dir, exist_ok=True)

random.seed(42)

# Verificar si hay checkpoint previo para continuar
if os.path.exists(OUTPUT_CKPT) and os.path.getsize(OUTPUT_CKPT) > 1e6:
    sd = torch.load(OUTPUT_CKPT, map_location='cpu')
    if 'model' in sd:
        sd = sd['model']
    sam2_model.load_state_dict(sd, strict=False)
    print(f"Previous checkpoint loaded: {OUTPUT_CKPT}")
else:
    print("Starting training from base checkpoint")

# Cuántos tiles de validación evaluar en CADA epoch (para no frenar el training).
# La evaluación FINAL (después del último epoch) siempre usa el set de val completo.
VAL_SAMPLES_PER_EPOCH = min(40, len(val_ds))

# Loop de entrenamiento
print(f"\nStarting SAM2 fine-tune ({EPOCHS_SAM2} epochs)...")
print(f"{'='*60}")

sam2_model.train()
loss_history = []          # compat con el resto del notebook (solo train loss)
history = []                # historial completo: train_loss, val_loss, val_iou, val_dice
best_val_iou = -1.0

for epoch in range(EPOCHS_SAM2):
    sam2_model.train()
    epoch_loss = 0.0
    n = 0

    # Barajar orden de entrenamiento
    order = list(range(len(train_ds)))
    random.shuffle(order)

    for i, idx in enumerate(order):
        img, union, vid = train_ds[idx]
        if union.sum() == 0:
            continue

        optimizer.zero_grad()
        loss = forward_tile(predictor, img, union)
        if loss is None:
            continue

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n += 1

        # Progreso cada 50 tiles
        if (i + 1) % 50 == 0:
            avg = epoch_loss / max(n, 1)
            print(f"  Epoch {epoch+1}/{EPOCHS_SAM2} | Tile {i+1}/{len(train_ds)} | Loss: {avg:.4f}")

    # Loss promedio del epoch
    avg_loss = epoch_loss / max(n, 1)
    loss_history.append(avg_loss)

    # Validación (submuestra, para no alargar demasiado cada epoch)
    val_stats = evaluate_val(predictor, sam2_model, val_ds, max_samples=VAL_SAMPLES_PER_EPOCH)

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "val_loss": val_stats["loss"],
        "val_iou": val_stats["iou"],
        "val_dice": val_stats["dice"],
        "val_precision": val_stats["precision"],
        "val_recall": val_stats["recall"],
        "val_n_samples": val_stats["n_samples"],
    })

    # Guardar checkpoint del último epoch
    torch.save({'model': sam2_model.state_dict()}, OUTPUT_CKPT)
    os.makedirs('/kaggle/output', exist_ok=True)
    import shutil
    shutil.copy(OUTPUT_CKPT, KAGGLE_OUTPUT)

    # Guardar checkpoint del MEJOR epoch según val_iou
    if val_stats["iou"] is not None and val_stats["iou"] > best_val_iou:
        best_val_iou = val_stats["iou"]
        torch.save({'model': sam2_model.state_dict(), 'epoch': epoch + 1, 'val_iou': best_val_iou},
                    OUTPUT_CKPT_BEST)
        shutil.copy(OUTPUT_CKPT_BEST, KAGGLE_OUTPUT_BEST)
        best_marker = "  <- best val_iou so far"
    else:
        best_marker = ""

    val_str = (f"val_loss={val_stats['loss']:.4f} | val_IoU={val_stats['iou']:.4f} | "
               f"val_Dice={val_stats['dice']:.4f} | val_Precision={val_stats['precision']:.4f} | "
               f"val_Recall={val_stats['recall']:.4f}") if val_stats["loss"] is not None else "val: no data"
    print(f"Epoch {epoch+1}/{EPOCHS_SAM2} — train_loss={avg_loss:.4f} | {val_str}{best_marker}")

# Guardar historial completo (para graficar y para el reporte final)
with open(os.path.join(metrics_dir, 'sam2_training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

# Evaluación final: set de validación completo con el mejor checkpoint
if os.path.exists(OUTPUT_CKPT_BEST):
    sd_best = torch.load(OUTPUT_CKPT_BEST, map_location='cpu')['model']
    sam2_model.load_state_dict(sd_best, strict=False)
final_eval = evaluate_val(predictor, sam2_model, val_ds, max_samples=None)

with open(os.path.join(metrics_dir, 'sam2_eval_final.json'), 'w') as f:
    json.dump({
        "epochs_trained": EPOCHS_SAM2,
        "best_epoch": max(history, key=lambda h: (h["val_iou"] is not None, h["val_iou"] or -1))["epoch"],
        "val_loss_final": final_eval["loss"],
        "val_iou_final": final_eval["iou"],
        "val_dice_final": final_eval["dice"],
        "val_precision_final": final_eval["precision"],
        "val_recall_final": final_eval["recall"],
        "val_n_samples_final": final_eval["n_samples"],
        "train_loss_initial": loss_history[0],
        "train_loss_final": loss_history[-1],
    }, f, indent=2)

# Gráfica de curvas
try:
    import matplotlib.pyplot as plt
    epochs_x = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    axes[0].plot(epochs_x, [h["train_loss"] for h in history], label="Train loss")
    axes[0].plot(epochs_x, [h["val_loss"] for h in history], label="Val loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Dice loss")
    axes[0].set_title("Loss curve (error)"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_x, [h["val_iou"] for h in history], label="Val IoU", color="green")
    axes[1].plot(epochs_x, [h["val_dice"] for h in history], label="Val Dice", color="orange")
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
    axes[1].set_title("Val IoU / Dice\n(main segmentation metric)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs_x, [h["val_precision"] for h in history], label="Val Precision", color="steelblue")
    axes[2].plot(epochs_x, [h["val_recall"] for h in history], label="Val Recall", color="crimson")
    axes[2].set_ylim(0, 1)
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Score")
    axes[2].set_title("Pixel-level Precision / Recall\n(diagnosis: over-segmenting vs. missing trees)")
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(metrics_dir, 'sam2_training_curves.png')
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"\nChart saved to: {plot_path}")
except Exception as e:
    print(f"Could not generate chart: {e}")

print(f"\n{'='*60}")
print(f"Fine-tune complete")
print(f"Last epoch checkpoint: {OUTPUT_CKPT}")
print(f"Best epoch checkpoint (by val IoU): {OUTPUT_CKPT_BEST}")
print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Loss improvement: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")
print(f"Final val IoU (full set, best checkpoint): {final_eval['iou']:.4f}" if final_eval['iou'] is not None else "Final val IoU: no data")
print(f"Final val Dice (full set, best checkpoint): {final_eval['dice']:.4f}" if final_eval['dice'] is not None else "Final val Dice: no data")
print(f"Final val Precision (full set, best checkpoint): {final_eval['precision']:.4f}" if final_eval['precision'] is not None else "Final val Precision: no data")
print(f"Final val Recall (full set, best checkpoint): {final_eval['recall']:.4f}" if final_eval['recall'] is not None else "Final val Recall: no data")

# --- Diagnóstico: ¿fueron suficientes los EPOCHS_SAM2 configurados? ---
print(f"\n{'='*60}")
print("DIAGNOSIS: were the configured epochs enough?")
print(f"{'='*60}")
n_history = len(history)
window = max(3, n_history // 5)  # último ~20% de epochs
valid_val_ious = [h["val_iou"] for h in history if h["val_iou"] is not None]
if len(valid_val_ious) >= window + 1:
    recent_improvement = valid_val_ious[-1] - valid_val_ious[-1 - window]
    print(f"Change in val_IoU between epoch {n_history - window} and epoch {n_history}: {recent_improvement:+.4f}")
    if recent_improvement > 0.01:
        print("val_IoU was still clearly rising at the end -> "
              "more epochs are probably worth it (or lower the learning rate and keep training).")
    elif recent_improvement < -0.01:
        print("val_IoU dropped toward the end -> sign of overfitting; "
              "use the 'best' checkpoint (already saved) instead of the last one, or reduce EPOCHS_SAM2.")
    else:
        print(f"val_IoU stabilized over the last {window} epochs -> "
              f"{EPOCHS_SAM2} epochs look sufficient for this dataset.")
else:
    print("Not enough validation points for a reliable automatic diagnosis; "
          "check the training curves chart saved in outputs/metrics/.")


### 7.1 Recargar sin re-entrenar (solo si perdiste las variables en memoria)

Usa esta celda SOLO si la sesión se reinició y `classifier`/`sam2_model` ya no
existen (por ejemplo, si al correr la celda de abajo te da `NameError`). Si
siguen en memoria, sáltatela — no hace nada malo si la corres de más, pero es
innecesaria.

No re-entrena nada: carga el clasificador (`crown_model.pkl`) y el mejor
checkpoint de SAM2 (`sam2_finetuned_best.pt`) ya guardados en disco.

In [ ]:
# Recargar sin re-entrenar
import os, pickle
import torch

# 1. Clasificador
try:
    classifier
    print("Classifier already exists in memory.")
except NameError:
    classifier_dir = os.path.join(PROJECT_ROOT, 'outputs', 'classifier')
    with open(os.path.join(classifier_dir, 'crown_model.pkl'), 'rb') as f:
        classifier = pickle.load(f)
    print("Classifier loaded from crown_model.pkl")

# 2. Cliente CVAT
try:
    client
    print("Client already exists in memory.")
except NameError:
    client = CvatClient(
        os.environ['CVAT_HOST'], os.environ['CVAT_USER'], os.environ['CVAT_PASSWORD']
    )
    print("Client re-authenticated")

# 3. SAM2
try:
    sam2_model
    print("Sam2_model already exists -- verifying that it has the fine-tuned weights...")
except NameError:
    sam2_model = build_sam2(SAM2_CONFIG, BASE_CKPT, device='cuda')
    predictor = SAM2ImagePredictor(sam2_model)
    print("Implemented SAM2 architecture.")

OUTPUT_CKPT_BEST = os.path.join(CKPT_DIR, 'sam2_finetuned_best.pt')
sd_best = torch.load(OUTPUT_CKPT_BEST, map_location='cpu')['model']
sam2_model.load_state_dict(sd_best, strict=False)
sam2_model.eval()
print("Fine-tuned weights loaded into sam2_model (sam2_finetuned_best.pt).")

print("\nReady—classifier, sam2_model, and client are ready, without retraining anything.")

---

## 7. Inferencia (Auto-etiquetado) por Finca

Ejecutar el pipeline de inferencia, finca por finca, pero **solo sobre los
frames que la Sección 4 marcó como `unlabeled_frames`**. Los frames que ya
tienen anotaciones manuales en CVAT no se tocan. 

Por cada finca con frames pendientes:

1. **Frames pendientes**: identificados en la Sección 4 (`LABELING_STATUS`); se
   descargan aquí mismo, solo los de esta finca (la Sección 3 no los bajó)
2. **SAM2 fine-tuned** → máscaras de segmentación
3. **Clasificador de copas** → crop_tree vs background
4. **NMS** → eliminar duplicados (IoU > 0.5)
5. **Filtros geométricos** → eliminar ruido
6. **Conversión a polígonos** → formato CVAT
7. **Guardar predicciones** → pred.json + overlays

Las fincas que ya están 100% etiquetadas se omiten automáticamente (no se re-infiere
sobre frames ya anotados).

**Limpieza automática**: justo después de cada finca (se haya inferido o se haya
omitido por estar completa), se borran sus frames crudos descargados
(`data/raw/<finca>/frames/`) si `CLEANUP_RAW_FRAMES_AFTER_USE = True`. Para ese punto
ya no se vuelven a necesitar — evita acumular miles de archivos en `/kaggle/working`,
que Kaggle solo persiste hasta ~500 archivos / ~20 GB al parar la sesión. No borra
ninguna predicción, checkpoint ni métrica; esas ya quedaron guardadas en otras carpetas
antes de la limpieza. Los overlays de inferencia sí se conservan (limitados a
`MAX_OVERLAYS_PER_FARM` por finca) para que puedas revisarlos.

In [ ]:
def nms_masks(masks, iou_thresh=0.5):
    """
    Non-Maximum Suppression para máscaras binarias.
    Elimina máscaras con alto solapamiento (IoU > threshold).
    """
    if not masks:
        return []

    areas = [m.sum() for m in masks]
    order = sorted(range(len(masks)), key=lambda i: areas[i], reverse=True)
    keep = []
    suppressed = set()

    for i in order:
        if i in suppressed:
            continue
        keep.append(i)
        for j in order:
            if j in suppressed or j == i:
                continue
            inter = np.logical_and(masks[i], masks[j]).sum()
            union = np.logical_or(masks[i], masks[j]).sum()
            if union > 0 and inter / union > iou_thresh:
                suppressed.add(j)

    return [masks[i] for i in keep]


def area_px_for_diameter_m(gsd_cm_px, diameter_m):
    """
    Convierte un diámetro real de copa (en metros) a área en píxeles,
    según el GSD (resolución) de una finca puntual.

    Esto es lo que evita que fincas con GSD alto (peor resolución, ej.
    Playa_Rica a 9.72 cm/px) pierdan árboles reales solo por ocupar pocos
    píxeles -- un umbral de área FIJO en píxeles (como MIN_AREA_POLYGON)
    no tiene en cuenta que la misma copa real ocupa muchos menos píxeles
    en una finca de peor resolución.
    """
    diam_px = (diameter_m * 100) / gsd_cm_px
    radius_px = diam_px / 2
    return 3.14159265 * radius_px ** 2


def masks_to_polygons(mask, min_area=None):
    """
    Convierte máscara binaria a lista de polígonos.

    min_area: si se da, reemplaza al MIN_AREA_POLYGON global para esta
    llamada -- se usa para pasar un umbral ajustado por GSD, finca por finca.
    """
    if min_area is None:
        min_area = MIN_AREA_POLYGON

    mask_uint8 = mask.astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        pts = cnt.reshape(-1, 2)
        if len(pts) >= 3:  # Mínimo 3 puntos para polígono
            polygons.append(pts.flatten().tolist())
    return polygons


def geometric_filter(polygons, min_area=900, max_area=50000, max_aspect=3.0):
    """
    Filtra polígonos por reglas geométricas:
    - Área mínima/máxima
    - Aspect ratio no extremo
    """
    filtered = []
    for pts in polygons:
        arr = np.array(pts, dtype=np.float32).reshape(-1, 2)
        x_min, y_min = arr.min(axis=0)
        x_max, y_max = arr.max(axis=0)
        w = x_max - x_min
        h = y_max - y_min

        if w <= 0 or h <= 0:
            continue

        # Calcular área usando shoelace
        x, y = arr[:, 0], arr[:, 1]
        area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

        aspect = max(w, h) / min(w, h) if min(w, h) > 0 else 999

        if area < min_area or area > max_area:
            continue
        if aspect > max_aspect:
            continue

        filtered.append(pts)

    return filtered

print("Post-processing functions defined (area filters now GSD-aware).")

In [ ]:
def infer_farm(farm_name, task_id, label_id, sam2_model, clf,
                   client, project_root, target_frames=None, save_overlays=True,
                   max_workers=8, max_overlays=None, gsd_cm_px=None):
    """
    Ejecuta el pipeline completo de inferencia para una finca.

    Pasos:
    1. Descargar frames desde CVAT
    2. SAM2 → máscaras de segmentación
    3. Clasificador → crop_tree/background
    4. NMS → eliminar duplicados
    5. Filtros geométricos
    6. Conversión a polígonos
    7. Guardar predicciones

    """
    print(f"\n{'='*60}")
    print(f"INFERENCE: {farm_name} (Task {task_id})")
    print(f"{'='*60}")

    if gsd_cm_px is not None:
        min_area_farm = area_px_for_diameter_m(gsd_cm_px, MIN_CROWN_DIAMETER_M)
        print(f"GSD: {gsd_cm_px} cm/px -> área mínima de árbol: {min_area_farm:.0f}px² "
              f"(antes fijo en {MIN_AREA_POLYGON}px² para todas las fincas)")
    else:
        min_area_farm = MIN_AREA_POLYGON
    max_area_farm = 50000  # fijo para todas las fincas

    # 1. Descargar los frames que hay que procesar (target_frames
    frame_dir = Path(project_root) / 'data' / 'raw' / str(task_id) / 'frames'
    frame_dir.mkdir(parents=True, exist_ok=True)

    if target_frames is not None:
        target_frames = set(target_frames)
        missing = [fn for fn in target_frames
                   if not (frame_dir / f'frame_{fn:05d}.png').exists()]
        if missing:
            print(f"Downloading {len(missing)} target frame(s) from CVAT...")
            try:
                cvat_frames = client.get_frames(task_id)
                frames_to_download = [f for f in cvat_frames if f['id'] in target_frames]
                client.download_frames_parallel(
                    task_id, frames_to_download, frame_dir, max_workers=max_workers,
                    desc="Downloading target frames"
                )
            except Exception as e:
                print(f"Error downloading frames (after retries): {e}")
                print(f"{farm_name}: skipping this farm due to download failure "
                      f"(will not crash the rest of the run).")
                return [], {
                    'farm': farm_name, 'task_id': task_id,
                    'frames_processed': 0, 'total_shapes': 0, 'stats': {},
                    'error': f'download failed: {e}'
                }
        frames = sorted(frame_dir.glob('frame_*.png'))
        frames = [f for f in frames if int(f.stem.split('_')[1]) in target_frames]
    else:
        existing_frames = sorted(frame_dir.glob('frame_*.png'))
        if len(existing_frames) == 0:
            print("Downloading all frames from CVAT (no target_frames given)...")
            try:
                cvat_frames = client.get_frames(task_id)
                client.download_frames_parallel(
                    task_id, cvat_frames, frame_dir, max_workers=max_workers,
                    desc="Downloading frames"
                )
            except Exception as e:
                print(f"Error downloading frames (after retries): {e}")
                print(f"{farm_name}: skipping this farm due to download failure "
                      f"(will not crash the rest of the run).")
                return [], {
                    'farm': farm_name, 'task_id': task_id,
                    'frames_processed': 0, 'total_shapes': 0, 'stats': {},
                    'error': f'download failed: {e}'
                }
        frames = sorted(frame_dir.glob('frame_*.png'))

    print(f"Unlabeled frames to process: {len(frames)}")

    if len(frames) == 0:
        print(f"{farm_name}: no frames pending labeling. Skipping inference.")
        return [], {
            'farm': farm_name, 'task_id': task_id,
            'frames_processed': 0, 'total_shapes': 0, 'stats': {}
        }

    # 2-6. Procesar cada frame
    out_dir = Path(project_root) / 'outputs' / 'pred' / farm_name.lower()
    out_dir.mkdir(parents=True, exist_ok=True)

    if save_overlays:
        overlay_dir = out_dir / 'overlays'
        overlay_dir.mkdir(exist_ok=True)

    # Elegir aleatoria los frames que tendrán overlay 
    overlay_target_frames = None
    if save_overlays and max_overlays is not None:
        overlay_rng = random.Random(f"overlays-{task_id}")
        frame_numbers_all = [int(f.stem.split('_')[1]) for f in frames]
        k = min(max_overlays, len(frame_numbers_all))
        overlay_target_frames = set(overlay_rng.sample(frame_numbers_all, k)) if k > 0 else set()

    all_shapes = []
    n_overlays_saved = 0
    n_frames_no_binary_mask = 0  # frames donde SAM2 no propuso NADA (score bajo o área total insuficiente)
    n_frames_zero_polygons = 0   # frames donde SAM2 sí propuso algo, pero no sobrevivió ningún polígono
    total_stats = {
        'total_detected': 0,
        'crop_tree': 0,
        'background': 0,
        'after_nms': 0,
        'after_filter': 0,
    }

    for i, fpath in enumerate(tqdm(frames, desc=f"Processing {farm_name}")):
        img_bgr = cv2.imread(str(fpath))
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]
        frame_no = int(fpath.stem.split('_')[1]) if '_' in fpath.stem else i

        # 2. SAM2 segmentation
        assert (h, w) == (512, 512), f"frame {frame_no}: Expected 512x512, received: {(h, w)}"
        predictor.reset_predictor()
        predictor.set_image(img_rgb)
        box = np.array([0, 0, w * w / 1024, h * h / 1024], dtype=np.float32)
        masks, scores, _ = predictor.predict(
            point_coords=None, point_labels=None, box=box, multimask_output=False)
        binary_mask = masks[0]
        binary_score = float(scores[0])
        
        if binary_score <= 0.5:
            n_frames_low_score += 1
            continue
        if binary_mask.sum() <= min_area_farm:
            n_frames_small_area += 1
            continue

        # Separar la máscara binaria en instancias (componentes conexas) y
        # reconstruir una máscara individual por cada una (para clasificar y
        # hacer NMS instancia por instancia).
        raw_polygons = masks_to_polygons((binary_mask > 0).astype(np.uint8), min_area=min_area_farm)
        total_stats['total_detected'] += len(raw_polygons)

        if len(raw_polygons) == 0:
            n_frames_zero_polygons += 1
            continue

        valid_masks = [polygon_to_mask(pts, w, h) for pts in raw_polygons]
        valid_masks = [m for m in valid_masks if m.sum() > 0]

        if len(valid_masks) == 0:
            continue

        # 3. Clasificar crop_tree/background
        crops = extract_crops(img_rgb, valid_masks, frame_no=frame_no)
        if len(crops) > 0:
            X = np.stack([extract_features(c['image'], c['mask']) for c in crops])
            preds = clf.predict(X)

            # Filtrar solo crop_tree (descartar background)
            crop_tree_masks = [m for m, p in zip(valid_masks, preds) if p == 1]
            total_stats['crop_tree'] += len(crop_tree_masks)
            total_stats['background'] += len(valid_masks) - len(crop_tree_masks)
        else:
            crop_tree_masks = valid_masks

        if len(crop_tree_masks) == 0:
            continue

        # 4. NMS
        nms_result_masks = nms_masks(crop_tree_masks, iou_thresh=0.5)
        total_stats['after_nms'] += len(nms_result_masks)

        # 5-6. Conversión a polígonos y filtros
        frame_shapes_start = len(all_shapes)
        for mask in nms_result_masks:
            polygons = masks_to_polygons(mask, min_area=min_area_farm)
            polygons = geometric_filter(polygons, min_area=min_area_farm, max_area=max_area_farm)

            for pts in polygons:
                shape = {
                    'type': 'polygon',
                    'label_id': label_id,
                    'frame': frame_no,
                    'points': pts,
                    'source': 'auto',
                    'occluded': False,
                    'z_order': 0,
                    'group': 0,
                    'attributes': [],
                }
                all_shapes.append(shape)
                total_stats['after_filter'] += 1

        # 7. Guardar overlay (muestra aleatoria de hasta max_overlays por finca)
        frame_shapes = all_shapes[frame_shapes_start:]  # solo los de este frame
        en_muestra_aleatoria = overlay_target_frames is None or frame_no in overlay_target_frames
        if save_overlays and frame_shapes and en_muestra_aleatoria:
            ov = img_bgr.copy()
            for s in frame_shapes:
                pts = np.array(s['points'], dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(ov, [pts], (0, 255, 0))
                cv2.polylines(ov, [pts], True, (0, 200, 0), 2)
            alpha = 0.4
            overlay = cv2.addWeighted(img_bgr, 1 - alpha, ov, alpha, 0)
            cv2.imwrite(str(overlay_dir / f'frame_{frame_no:05d}_overlay.png'), overlay)
            n_overlays_saved += 1

    if save_overlays and max_overlays is not None and n_overlays_saved < max_overlays:
        frames_with_detections = sorted({s['frame'] for s in all_shapes})
        already_tried = overlay_target_frames or set()
        remaining_candidates = [fn for fn in frames_with_detections if fn not in already_tried]
        fallback_rng = random.Random(f"overlays-fallback-{task_id}")
        fallback_rng.shuffle(remaining_candidates)
        needed = max_overlays - n_overlays_saved

        for frame_no in remaining_candidates[:needed]:
            frame_path = frame_dir / f'frame_{frame_no:05d}.png'
            if not frame_path.exists():
                continue
            img_bgr = cv2.imread(str(frame_path))
            if img_bgr is None:
                continue
            frame_shapes = [s for s in all_shapes if s['frame'] == frame_no]
            ov = img_bgr.copy()
            for s in frame_shapes:
                pts = np.array(s['points'], dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(ov, [pts], (0, 255, 0))
                cv2.polylines(ov, [pts], True, (0, 200, 0), 2)
            alpha = 0.4
            overlay = cv2.addWeighted(img_bgr, 1 - alpha, ov, alpha, 0)
            cv2.imwrite(str(overlay_dir / f'frame_{frame_no:05d}_overlay.png'), overlay)
            n_overlays_saved += 1

        if n_overlays_saved == 0 and all_shapes:
            print(f"NOTICE: there was {len(all_shapes)} Shapes were detected, but it was not possible "
                  f"save any overlay (check frame_dir).")

    # Guardar predicciones
    pred_data = {'shapes': all_shapes}
    with open(out_dir / 'pred.json', 'w') as f:
        json.dump(pred_data, f, indent=2)

    # Guardar manifest
    manifest = {
        'farm': farm_name,
        'task_id': task_id,
        'frames_processed': len(frames),
        'frames_no_binary_mask': n_frames_no_binary_mask,
        'frames_zero_polygons': n_frames_zero_polygons,
        'total_shapes': len(all_shapes),
        'overlays_saved': n_overlays_saved,
        'stats': total_stats,
    }
    with open(out_dir / 'pred_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)

    # Resumen
    print(f"\n{'='*60}")
    print(f"SUMMARY: {farm_name}")
    print(f"{'='*60}")
    print(f"Frames processed: {len(frames)}")
    print(f"Frames where SAM2 proposed NOTHING (score<=0.5 or area too small): "
          f"{n_frames_no_binary_mask}/{len(frames)}")
    print(f"Frames where SAM2 proposed something but 0 polygons survived: {n_frames_zero_polygons}")
    print(f"Total shapes detected: {total_stats['total_detected']}")
    print(f"Crop_tree: {total_stats['crop_tree']}")
    print(f"Background: {total_stats['background']}")
    print(f"After NMS: {total_stats['after_nms']}")
    print(f"After filters: {total_stats['after_filter']}")
    print(f"Predictions saved to: {out_dir / 'pred.json'}")

    return all_shapes, manifest

print("infer_farm() function defined.")


def cleanup_farm_raw_frames(project_root, task_id):
    """
    Borra los frames crudos (PNG) ya descargados de una finca, una vez que
    tanto el entrenamiento (Secciones 5-7) como la inferencia (Sección 8) ya
    los usaron. Se conservan manifest.json y annotations.json (livianos, sirven
    de registro/auditoría).
    """
    import shutil
    frame_dir = Path(project_root) / 'data' / 'raw' / str(task_id) / 'frames'
    if not frame_dir.exists():
        return 0, 0.0

    files = list(frame_dir.glob('*.png'))
    n_files = len(files)
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    shutil.rmtree(frame_dir)
    return n_files, size_mb

In [ ]:
# Prueba: solo La_Carolina (76 frames), antes de lanzar las 15 fincas
print("="*60)
print("Test — La_Carolina, with GSD-aware filter")
print("="*60)

test_farm = "La_Carolina"
test_cfg = FARMS[test_farm]

# Capturar el resultado anterior (antes del fix), antes de que se sobrescriba
test_manifest_path = Path(PROJECT_ROOT) / 'outputs' / 'pred' / test_farm.lower() / 'pred_manifest.json'
shapes_antes = 0
if test_manifest_path.exists():
    shapes_antes = json.loads(test_manifest_path.read_text()).get('total_shapes', 0)
print(f"Shapes detected before (fixed threshold 900px: {shapes_antes}")

test_pending_frames = LABELING_STATUS[test_farm]['unlabeled_frames']
print(f"Configured GSD: {test_cfg.get('gsd_cm_px')} cm/px")
print(f"Pending frames: {len(test_pending_frames)}")

test_shapes, test_manifest = infer_farm(
    farm_name=test_farm,
    task_id=test_cfg["task_id"],
    label_id=test_cfg["label_id"],
    sam2_model=sam2_model,
    clf=classifier,
    client=client,
    project_root=PROJECT_ROOT,
    target_frames=test_pending_frames,
    save_overlays=True,
    max_workers=DOWNLOAD_MAX_WORKERS,
    max_overlays=MAX_OVERLAYS_PER_FARM,
    gsd_cm_px=test_cfg.get('gsd_cm_px')
)

print("\n" + "="*60)
print("Test result")
print("="*60)
print(f"Shapes detected before:   {shapes_antes}")
print(f"Shapes detected now:   {test_manifest['total_shapes']}")
print(f"Difference: {test_manifest['total_shapes'] - shapes_antes:+d}")
print(f"Detailed stats: {test_manifest['stats']}")

if CLEANUP_RAW_FRAMES_AFTER_USE:
    n_files, size_mb = cleanup_farm_raw_frames(PROJECT_ROOT, test_cfg["task_id"])
    if n_files:
        print(f"\nCleanup: removed {n_files} raw frame files ({size_mb:.1f} MB) for {test_farm}")

print("\nCheck the overlays in outputs/pred/la_carolina/overlays/.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

overlay_dir = Path(PROJECT_ROOT) / 'outputs' / 'pred' / 'la_carolina' / 'overlays'
for fpath in sorted(overlay_dir.glob('*.png')):
    img = mpimg.imread(fpath)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(fpath.name)
    plt.axis('off')
    plt.show()

In [ ]:
# Inferencia (auto-etiquetado) sobre las 15 fincas.
print("="*60)
print("AUTOMATIC INFERENCE — UNLABELED FRAMES (15 FARMS, GSD-AWARE FILTER)")
print("="*60)

# Capturar el resultado de la corrida anterior (antes de este re-run), para
# poder comparar cuántas anotaciones se ganan con el filtro nuevo, por finca.
shapes_before_gsd_fix = {}
pred_root = Path(PROJECT_ROOT) / 'outputs' / 'pred'
if pred_root.exists():
    for farm_dir in pred_root.iterdir():
        manifest_path = farm_dir / 'pred_manifest.json'
        if manifest_path.exists():
            old_manifest = json.loads(manifest_path.read_text())
            shapes_before_gsd_fix[old_manifest['farm']] = old_manifest.get('total_shapes', 0)
print(f"Snapshot 'antes' capturado para {len(shapes_before_gsd_fix)} finca(s).")

inference_results = {}
total_cleaned_files = 0
total_cleaned_mb = 0.0

for name, cfg in FARMS.items():
    pending_frames = LABELING_STATUS[name]['unlabeled_frames']

    if not pending_frames:
        print(f"\n{name}: 0 pending frames (farm already 100% labeled). Skipping.")
    else:
        shapes, manifest = infer_farm(
            farm_name=name,
            task_id=cfg["task_id"],
            label_id=cfg["label_id"],
            sam2_model=sam2_model,
            clf=classifier,
            client=client,
            project_root=PROJECT_ROOT,
            target_frames=pending_frames,
            save_overlays=True,
            max_workers=DOWNLOAD_MAX_WORKERS,
            max_overlays=MAX_OVERLAYS_PER_FARM,
            gsd_cm_px=cfg.get('gsd_cm_px')
        )
        inference_results[name] = manifest

    # Limpieza: para este punto del pipeline, los frames crudos de esta finca
    # ya se usaron en preparación de dataset (Sección 5), clasificador
    # (Sección 6) y, si aplicaba, inferencia (arriba) — no se vuelven a leer
    # en ningún paso posterior (upload y reporte solo leen JSON/manifests).
    if CLEANUP_RAW_FRAMES_AFTER_USE:
        n_files, size_mb = cleanup_farm_raw_frames(PROJECT_ROOT, cfg["task_id"])
        if n_files:
            total_cleaned_files += n_files
            total_cleaned_mb += size_mb
            print(f"  Cleanup: removed {n_files} raw frame files ({size_mb:.1f} MB) for {name}")

print("\n" + "="*60)
print(f"Inference completed for {len(inference_results)} farm(s) with pending frames.")
if CLEANUP_RAW_FRAMES_AFTER_USE:
    print(f"Cleanup total: {total_cleaned_files} files removed ({total_cleaned_mb:.1f} MB freed)")
print("="*60)

### 8.1 Revisión visual — overlays por finca

Muestra en pantalla los overlays de ejemplo que se guardaron para cada finca
(hasta `MAX_OVERLAYS_PER_FARM` por finca, elegidos al azar entre los frames
procesados). Las fincas sin frames pendientes o sin detecciones no muestran nada.

In [ ]:
# Mostrar los overlays de ejemplo de cada finca (los mismos ya guardados en disco)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for name in FARMS.keys():
    overlay_dir = Path(PROJECT_ROOT) / 'outputs' / 'pred' / name.lower() / 'overlays'
    if not overlay_dir.exists():
        continue
    overlay_files = sorted(overlay_dir.glob('*.png'))
    if not overlay_files:
        continue

    n = len(overlay_files)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, fpath in zip(axes, overlay_files):
        img = mpimg.imread(fpath)
        ax.imshow(img)
        ax.set_title(fpath.stem, fontsize=9)
        ax.axis('off')
    fig.suptitle(f"{name} — {n} overlay(s) de ejemplo", fontsize=14)
    plt.tight_layout()
    plt.show()

### 8.2 Anotaciones por finca — antes vs. después del filtro GSD-aware

Compara el número de anotaciones (shapes) generadas en la corrida anterior
(umbral fijo de 900px) contra esta corrida (umbral ajustado por GSD real de
cada finca). Las fincas ya 100% etiquetadas no cambian (no se auto-etiquetan).

In [ ]:
# Gráfica: número de anotaciones por finca, antes vs. después del filtro GSD-aware
import matplotlib.pyplot as plt
import numpy as np

names = list(FARMS.keys())
before_vals = [shapes_before_gsd_fix.get(n, 0) for n in names]
after_vals = [inference_results[n]['total_shapes'] if n in inference_results
              else shapes_before_gsd_fix.get(n, 0)
              for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - width/2, before_vals, width, label='Antes (umbral fijo 900px)', color='steelblue')
ax.bar(x + width/2, after_vals, width, label='Después (umbral GSD-aware)', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('Número de anotaciones (shapes)')
ax.set_title('Anotaciones por finca: antes vs. después del filtro de área ajustado por GSD')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

metrics_dir = os.path.join(PROJECT_ROOT, 'outputs', 'metrics')
os.makedirs(metrics_dir, exist_ok=True)
chart_path = os.path.join(metrics_dir, 'anotaciones_antes_despues_gsd.png')
plt.savefig(chart_path, dpi=150)
plt.show()

total_before = sum(before_vals)
total_after = sum(after_vals)
diff_pct = (total_after / total_before - 1) * 100 if total_before else 0
print(f"\nGráfica guardada en: {chart_path}")
print(f"Total anotaciones ANTES: {total_before}")
print(f"Total anotaciones DESPUÉS: {total_after}")
print(f"Diferencia: {total_after - total_before:+d} ({diff_pct:+.1f}%)")

---

## 8. Consolidación y Upload a CVAT

Consolidamos los resultados de **todas las fincas que tuvieron frames pendientes**
de etiquetar (las que ya estaban 100% etiquetadas no generan predicciones nuevas,
así que no aparecen aquí) y opcionalmente subimos las predicciones a CVAT.

**IMPORTANTE**: El upload está **deshabilitado por defecto** (`UPLOAD_TO_CVAT = False`).
Solo se ejecutará si cambias la variable a `True` y confirmas la autorización.

### Gate de seguridad:
1. Validar payload (< 120 shapes/frame)
2. Verificar overlap con frames existentes
3. Requiere autorización explícita

In [ ]:
def validate_payload(pred_path, max_shapes_per_frame=120):
    """
    Valida el payload antes de subir a CVAT.
    Verifica que no haya más de max_shapes_per_frame por frame.
    """
    data = json.loads(Path(pred_path).read_text())
    shapes = data.get('shapes', [])

    # Contar shapes por frame
    from collections import Counter
    per_frame = Counter(s['frame'] for s in shapes)

    # Verificar límite
    problems = []
    for frame, count in per_frame.items():
        if count > max_shapes_per_frame:
            problems.append(f"Frame {frame}: {count} shapes (limit: {max_shapes_per_frame})")

    if problems:
        print("ERROR: Payload exceeds limits:")
        for p in problems:
            print(f"  - {p}")
        return False

    print(f"Valid payload: {len(shapes)} shapes across {len(per_frame)} frames")
    return True

def upload_to_cvat(pred_path, task_id, client, authorize=False):
    """
    Sube predicciones a CVAT.

    Gate de seguridad:
    1. Validar payload
    2. Verificar overlap con frames existentes
    3. Requiere autorización explícita

    Endpoint: PATCH /api/tasks/{id}/annotations?action=create — es el endpoint
    real de la API de CVAT para AGREGAR shapes sin borrar lo que ya existe
    (POST /api/tasks/{id}/shapes, usado antes, no es un endpoint válido de CVAT).
    Se envía el 'version' actual de las anotaciones para evitar pisar cambios
    hechos por otra persona en paralelo.

    RECOMENDADO: probar primero con UPLOAD_TO_CVAT=True en UNA sola finca chica
    antes de confiar en que sube bien en las 15.
    """
    if not authorize:
        print("\n" + "="*60)
        print("UPLOAD NOT EXECUTED")
        print("="*60)
        print("UPLOAD_TO_CVAT is False.")
        print("To upload, change UPLOAD_TO_CVAT = True and re-run this cell.")
        return

    print(f"\n{'='*60}")
    print(f"UPLOAD TO CVAT: Task {task_id}")
    print(f"{'='*60}")

    # 1. Validar payload
    if not validate_payload(pred_path):
        print("ABORTED: Invalid payload.")
        return

    # 2. Cargar shapes
    data = json.loads(Path(pred_path).read_text())
    shapes = data.get('shapes', [])

    if not shapes:
        print("Nothing to upload (0 shapes).")
        return

    # 3. Verificar overlap con frames existentes + obtener version actual
    current_version = None
    try:
        annotations = client.get_annotations(task_id)
        current_version = annotations.get('version')
        already_annotated_frames = set()
        for shape in annotations.get('shapes', []):
            already_annotated_frames.add(shape.get('frame', -1))

        new_frames = {s['frame'] for s in shapes}
        overlap = new_frames & already_annotated_frames

        if overlap:
            print(f"WARNING: Overlap detected in frames: {sorted(overlap)}")
            print("These frames already have annotations in CVAT.")
            print("Continuing with upload (additional shapes will be added)...")
    except Exception as e:
        print(f"Could not verify overlap/version: {e}")

    # 4. Subir shapes vía PATCH .../annotations?action=create
    print(f"\nUploading {len(shapes)} shapes to task {task_id}...")
    try:
        url = f"{client.host}/api/tasks/{task_id}/annotations"
        params = {'action': 'create'}

        # Subir en lotes de 100
        batch_size = 100
        for i in range(0, len(shapes), batch_size):
            batch = shapes[i:i+batch_size]
            payload = {'shapes': batch, 'tags': [], 'tracks': []}
            if current_version is not None:
                payload['version'] = current_version

            resp = client._request('PATCH', url, params=params, json=payload)
            print(f"  Batch {i//batch_size + 1}: {len(batch)} shapes uploaded")

            # La versión sube con cada escritura; refrescarla para el próximo lote
            try:
                current_version = resp.json().get('version', current_version)
            except Exception:
                pass

        print(f"\nUpload complete: {len(shapes)} shapes on task {task_id}")
    except Exception as e:
        print(f"Error during upload (after retries): {e}")

print("Upload functions defined (endpoint PATCH .../annotations?action=create).")

In [ ]:
# Consolidar resultados
print("="*60)
print("RESULTS CONSOLIDATION")
print("="*60)

total_shapes = 0
for farm, manifest in inference_results.items():
    n = manifest.get('total_shapes', 0)
    total_shapes += n
    print(f"{farm}: {n} shapes (over {manifest.get('frames_processed', 0)} unlabeled frames)")

print(f"\nTotal: {total_shapes} new shapes, across {len(inference_results)} farm(s)")

# Guardar resumen consolidado
summary_path = os.path.join(PROJECT_ROOT, 'outputs', 'inference_summary.json')
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, 'w') as f:
    json.dump(inference_results, f, indent=2)
print(f"Summary saved: {summary_path}")

In [ ]:
# Upload a CVAT (solo si UPLOAD_TO_CVAT = True)
print("="*60)
print("UPLOAD TO CVAT")
print("="*60)

# Prueba de subida de los datos (Primero una finca luego False todas las fincas)
TEST_UPLOAD_SINGLE_FARM = True

if not inference_results:
    print("No new predictions to upload (all farms were already labeled).")
else:
    farms_to_upload = list(inference_results.keys())
    if TEST_UPLOAD_SINGLE_FARM and UPLOAD_TO_CVAT and len(farms_to_upload) > 1:
        print(f"TEST_UPLOAD_SINGLE_FARM=True -> uploading only '{farms_to_upload[0]}' as a test.")
        print("Check in CVAT that it looks right, then set TEST_UPLOAD_SINGLE_FARM=False")
        print("and re-run this cell to upload the rest.\n")
        farms_to_upload = farms_to_upload[:1]

    for name in farms_to_upload:
        pred_path = os.path.join(PROJECT_ROOT, 'outputs', 'pred', name.lower(), 'pred.json')
        upload_to_cvat(
            pred_path,
            FARMS[name]['task_id'],
            client,
            authorize=UPLOAD_TO_CVAT
        )

---

## 9. Reporte de Métricas (para el artículo)

Consolidar en un solo lugar todos los resultados:

- **Dataset**: frames totales, etiquetados manualmente (antes) vs. auto-etiquetados
  (después), por finca y en total.
- **Anotaciones**: número de polígonos/etiquetas antes (manuales en CVAT) vs.
  después (manuales + generadas automáticamente).
- **Clasificador de copas**: accuracy, precisión, recall, F1 (si ya hay labels
  reales crop_tree/background; si no, se marca explícitamente como no disponible).
- **SAM2 fine-tuned**: loss inicial/final, IoU y Dice en validación.
- **Inferencia**: detecciones totales, crop_tree vs. background, y cuántas
  sobrevivieron el NMS y los filtros geométricos.

Todo se guarda en `outputs/final_report.json` (datos crudos) y
`outputs/final_report.md` (tabla en Markdown, lista para pegar en el artículo).

In [ ]:
# Reporte consolidado de métricas
print("="*60)
print("GENERATING FINAL METRICS REPORT")
print("="*60)

metrics_dir = os.path.join(PROJECT_ROOT, 'outputs', 'metrics')
os.makedirs(metrics_dir, exist_ok=True)

# 1. Dataset: antes vs. después, por finca
farm_reports = {}
total_frames = total_labeled_before = total_auto_labeled = 0
total_shapes_before = total_new_shapes = 0

for name, cfg in FARMS.items():
    info = LABELING_STATUS.get(name, {})
    n_total = info.get('total_frames', 0)
    n_labeled_before = len(info.get('labeled_frames', []))
    shapes_before = info.get('shapes_before', 0)

    manifest = inference_results.get(name)
    n_auto_labeled = manifest.get('frames_processed', 0) if manifest else 0
    new_shapes = manifest.get('total_shapes', 0) if manifest else 0

    n_labeled_after = n_labeled_before + n_auto_labeled
    shapes_after = shapes_before + new_shapes

    farm_reports[name] = {
        "crop_type": cfg["crop_type"],
        "total_frames": n_total,
        "frames_labeled_before": n_labeled_before,
        "frames_unlabeled_before": len(info.get('unlabeled_frames', [])),
        "frames_auto_labeled": n_auto_labeled,
        "frames_labeled_after": n_labeled_after,
        "coverage_before_pct": round(n_labeled_before / n_total * 100, 1) if n_total else 0.0,
        "coverage_after_pct": round(n_labeled_after / n_total * 100, 1) if n_total else 0.0,
        "shapes_before": shapes_before,
        "new_shapes": new_shapes,
        "shapes_after": shapes_after,
        "inference_stats": manifest.get('stats', {}) if manifest else {},
    }

    total_frames += n_total
    total_labeled_before += n_labeled_before
    total_auto_labeled += n_auto_labeled
    total_shapes_before += shapes_before
    total_new_shapes += new_shapes

total_labeled_after = total_labeled_before + total_auto_labeled
total_shapes_after = total_shapes_before + total_new_shapes

print(f"\nTotal frames (15 farms): {total_frames}")
print(f"Frames labeled BEFORE: {total_labeled_before} ({total_labeled_before/total_frames*100:.1f}%)")
print(f"Frames auto-labeled in this run: {total_auto_labeled}")
print(f"Frames labeled AFTER: {total_labeled_after} ({total_labeled_after/total_frames*100:.1f}%)")
print(f"Shapes (polygons) BEFORE: {total_shapes_before}")
print(f"New shapes (auto): {total_new_shapes}")
print(f"Shapes AFTER: {total_shapes_after}")

# 2. Clasificador de copas
clf_metrics_path = os.path.join(PROJECT_ROOT, 'outputs', 'classifier', 'metrics.json')
clf_metrics = json.loads(Path(clf_metrics_path).read_text()) if os.path.exists(clf_metrics_path) else {}

# 3. SAM2 fine-tuned
sam2_eval_path = os.path.join(metrics_dir, 'sam2_eval_final.json')
sam2_metrics = json.loads(Path(sam2_eval_path).read_text()) if os.path.exists(sam2_eval_path) else {}

#  4. Inferencia agregada (across fincas)
aggregated_inference = {"total_detected": 0, "crop_tree": 0, "background": 0, "after_nms": 0, "after_filter": 0}
for manifest in inference_results.values():
    for k in aggregated_inference:
        aggregated_inference[k] += manifest.get('stats', {}).get(k, 0)

# 5. Consolidar y guardar
final_report = {
    "dataset": {
        "farms": farm_reports,
        "totals": {
            "total_frames": total_frames,
            "frames_labeled_before": total_labeled_before,
            "frames_auto_labeled": total_auto_labeled,
            "frames_labeled_after": total_labeled_after,
            "coverage_before_pct": round(total_labeled_before / total_frames * 100, 1) if total_frames else 0.0,
            "coverage_after_pct": round(total_labeled_after / total_frames * 100, 1) if total_frames else 0.0,
            "shapes_before": total_shapes_before,
            "new_shapes": total_new_shapes,
            "shapes_after": total_shapes_after,
        },
    },
    "crown_classifier": clf_metrics,
    "sam2_finetuned": sam2_metrics,
    "aggregated_inference": aggregated_inference,
}

with open(os.path.join(PROJECT_ROOT, 'outputs', 'final_report.json'), 'w') as f:
    json.dump(final_report, f, indent=2)

# 6. Tabla en Markdown (lista para pegar en el artículo)
md_lines = []
md_lines.append("# Results Report — Fruit Tree Auto-labeling\n")
md_lines.append("## Labeling coverage by farm\n")
md_lines.append("| Farm | Crop | Frames | Labeled before | % before | Labeled after | % after | Shapes before | Shapes after |")
md_lines.append("|---|---|---|---|---|---|---|---|---|")
for name, r in farm_reports.items():
    md_lines.append(f"| {name} | {r['crop_type']} | {r['total_frames']} | "
                     f"{r['frames_labeled_before']} | {r['coverage_before_pct']}% | "
                     f"{r['frames_labeled_after']} | {r['coverage_after_pct']}% | "
                     f"{r['shapes_before']} | {r['shapes_after']} |")
t = final_report["dataset"]["totals"]
md_lines.append(f"| **TOTAL** | | {t['total_frames']} | {t['frames_labeled_before']} | "
                 f"{t['coverage_before_pct']}% | {t['frames_labeled_after']} | "
                 f"{t['coverage_after_pct']}% | {t['shapes_before']} | {t['shapes_after']} |")

md_lines.append("\n## Crown classifier (crop_tree vs. background)\n")
if clf_metrics.get("has_real_labels"):
    md_lines.append(f"- Accuracy (hold-out 20%): {clf_metrics.get('test_accuracy', 'N/A')}")
    md_lines.append(f"- Precision: {clf_metrics.get('test_precision', 'N/A')}")
    md_lines.append(f"- Recall: {clf_metrics.get('test_recall', 'N/A')}")
    md_lines.append(f"- F1-score: {clf_metrics.get('test_f1', 'N/A')}")
    md_lines.append(f"- CV accuracy: {clf_metrics.get('cv_accuracy', 'N/A')} ± {clf_metrics.get('cv_std', 'N/A')}")
    md_lines.append(f"- Average Precision (AP, PR curve): {clf_metrics.get('average_precision', 'N/A')}")
else:
    md_lines.append("- **Not available**: there are no real crop_tree/background human labels "
                     "in CVAT yet (all crops were marked 'crop_tree' by default). "
                     "This classifier's metrics are not valid for the article "
                     "until both classes are labeled.")

md_lines.append("\n## SAM2 fine-tuned\n")
if sam2_metrics:
    md_lines.append(f"- Epochs trained: {sam2_metrics.get('epochs_trained', 'N/A')}")
    md_lines.append(f"- Best epoch (by val IoU): {sam2_metrics.get('best_epoch', 'N/A')}")
    loss_init = sam2_metrics.get('train_loss_initial')
    loss_final = sam2_metrics.get('train_loss_final')
    loss_str = (f"{loss_init:.4f} → {loss_final:.4f}"
                if isinstance(loss_init, (int, float)) and isinstance(loss_final, (int, float))
                else "N/A")
    md_lines.append(f"- Initial → final train loss: {loss_str}")
    md_lines.append(f"- Val IoU (full set, best checkpoint): {sam2_metrics.get('val_iou_final', 'N/A')}")
    md_lines.append(f"- Val Dice (full set, best checkpoint): {sam2_metrics.get('val_dice_final', 'N/A')}")
    md_lines.append(f"- Pixel-level Val Precision (best checkpoint): {sam2_metrics.get('val_precision_final', 'N/A')}")
    md_lines.append(f"- Pixel-level Val Recall (best checkpoint): {sam2_metrics.get('val_recall_final', 'N/A')}")

md_lines.append("\n## Inference (auto-labeling) — aggregated across 15 farms\n")
md_lines.append(f"- Total objects detected: {aggregated_inference['total_detected']}")
md_lines.append(f"- Classified as crop_tree: {aggregated_inference['crop_tree']}")
md_lines.append(f"- Classified as background: {aggregated_inference['background']}")
md_lines.append(f"- After NMS (duplicates removed): {aggregated_inference['after_nms']}")
md_lines.append(f"- After geometric filters (final): {aggregated_inference['after_filter']}")

report_md_path = os.path.join(PROJECT_ROOT, 'outputs', 'final_report.md')
with open(report_md_path, 'w') as f:
    f.write("\n".join(md_lines))

print(f"\nJSON report saved to: {os.path.join(PROJECT_ROOT, 'outputs', 'final_report.json')}")
print(f"Markdown report saved to: {report_md_path}")

---

## Resumen Final

Pipeline completado exitosamente. Aquí está el resumen de lo ejecutado:

### Proceso ejecutado:
1. **Setup**: Verificación de GPU e instalación de dependencias
2. **Configuración**: Definición de las 15 fincas y parámetros (incluye tamaño de SAM2, workers de descarga, límite de tiles del clasificador)
3. **Descarga CVAT**: Frames y anotaciones de las 15 fincas, en paralelo y con reintentos automáticos
4. **Análisis de etiquetado**: detección, finca por finca, de frames y shapes etiquetados vs. sin etiquetar (snapshot "antes")
5. **Preparación Dataset**: Conversión a formato SAM2 SA-V, usando TODO frame ya etiquetado de las 15 fincas
6. **Clasificador**: RandomForest (crop_tree vs background), con negativos auto-generados y métricas hold-out (precisión/recall/F1)
7. **Fine-tune SAM2**: Entrenamiento del modelo de segmentación, con validación de IoU/Dice por epoch
8. **Inferencia**: Auto-etiquetado SOLO de los frames sin etiquetar, finca por finca (segmentación binaria + componentes conexas)
9. **Upload**: Subida a CVAT vía el endpoint correcto (PATCH .../annotations?action=create), con prueba en 1 finca antes de subir todas
10. **Reporte final**: Consolidación de todas las métricas (antes/después) para el artículo

### Mejoras aplicadas en esta revisión:
- **Bug crítico corregido**: el loop de inferencia indexaba mal la salida de SAM2 (`masks[j]`/`scores[j]` sobre una máscara ya colapsada). Ahora se trata como una máscara binaria por frame y se separa en instancias por componentes conexas, consistente con cómo se entrena el modelo.
- **Endpoint de upload corregido**: `POST .../shapes` (no es un endpoint real de CVAT) → `PATCH .../annotations?action=create`, con `version` para evitar sobreescribir cambios en paralelo.
- **Descarga en paralelo** de frames (`ThreadPoolExecutor`) en vez de uno por uno.
- **Reintentos automáticos con backoff** en todas las llamadas HTTP a CVAT.
- **Sin límite fijo de tiles** para el clasificador (`MAX_TILES_CLASSIFIER`, antes 200 fijo).
- **Contraseña hardcodeada eliminada** — ahora requiere Kaggle Secrets, sin fallback en texto plano.
- **Metadata liviana persistida en `/kaggle/output`** tras cada finca descargada (manifest + anotaciones), para no perder el progreso si la sesión se cae.
- **Tamaño de modelo SAM2 configurable** (`SAM2_MODEL_SIZE`: large/base_plus/small/tiny) para ajustar velocidad vs. precisión.
- **Salvavidas de upload**: por defecto solo sube 1 finca de prueba antes de subir las demás.
- **Limpieza automática de frames crudos** (`CLEANUP_RAW_FRAMES_AFTER_USE`): borra los PNG
  descargados de cada finca justo después de usarlos (Secciones 4-7), y limita los overlays
  guardados por finca (`MAX_OVERLAYS_PER_FARM`), para no acercarse al límite de ~500
  archivos que Kaggle persiste al parar la sesión o hacer commit. No afecta ninguna
  métrica: corre después de que predicciones/checkpoints/reportes ya quedaron guardados.

### Archivos generados:
- `data/processed/` — Dataset en formato SAM2 (a partir de todos los frames ya etiquetados)
- `outputs/classifier/crown_model.pkl` y `metrics.json` — Clasificador de copas + métricas
- `checkpoints/sam2_finetuned.pt` — SAM2 fine-tuned (último epoch)
- `checkpoints/sam2_finetuned_best.pt` — SAM2 fine-tuned (mejor epoch según val IoU)
- `outputs/metrics/sam2_training_history.json` — historial epoch a epoch (train loss, val loss, val IoU, val Dice)
- `outputs/metrics/sam2_eval_final.json` — métricas finales de SAM2 en validación
- `outputs/metrics/sam2_training_curves.png` — gráfica de curvas (para el artículo)
- `outputs/metrics/labeling_status_before.json` — snapshot de etiquetado ANTES del auto-etiquetado
- `outputs/pred/<finca>/` — Predicciones por finca (solo las que tenían frames pendientes)
- `outputs/inference_summary.json` — Resumen consolidado de inferencia
- `outputs/final_report.json` y `outputs/final_report.md` — **Reporte consolidado antes/después, listo para el artículo**
- `/kaggle/output/raw_meta/<task_id>/` — Metadata liviana de cada finca (sobrevive a caídas de sesión)

### Siguientes pasos:
1. Revisar los overlays en `outputs/pred/*/overlays/`
2. Revisar `outputs/final_report.md` y la gráfica de curvas para el artículo
3. Si los resultados son satisfactorios, cambiar `UPLOAD_TO_CVAT = True` (empieza subiendo solo 1 finca)
4. Verificar en CVAT que esa finca subió bien, poner `TEST_UPLOAD_SINGLE_FARM = False` y volver a correr la celda de upload

### Notas:
- El notebook es **100% independiente** — no requiere archivos externos
- Todos los comentarios están en **español**
- El entrenamiento de SAM2 usa TODOS los frames ya etiquetados de las 15 fincas (no solo 2 fijas)
- La inferencia respeta lo ya anotado en CVAT: nunca re-procesa frames que ya tienen etiqueta
- El upload está **gateado** por seguridad — requiere autorización explícita
- Las métricas del clasificador de copas dependen de que la generación automática de
  background haya funcionado (frames no demasiado densos); si falla, el reporte lo marca explícitamente
- Los frames crudos (`data/raw/*/frames/`) se borran finca por finca a medida que avanza
  la Sección 8; si quieres inspeccionarlos después de correr, pon `CLEANUP_RAW_FRAMES_AFTER_USE = False`
  antes de correr, o revisa los overlays guardados (`outputs/pred/*/overlays/`), que sí se conservan
